In [ ]:
# ============================================================
# Hormuz Impact Project
# ETL Module : Oil
# Dataset    : DS_OIL_01_Brent_Crude.csv
# Version    : 2.0 (Gold Layer)
# ============================================================

import pandas as pd
import numpy as np
from google.colab import files

# ============================================================
# STEP 1 : Upload Dataset
# ============================================================

uploaded = files.upload()
filename = list(uploaded.keys())[0]

# ============================================================
# STEP 2 : Read Dataset
# ============================================================

df = pd.read_csv(filename)

print("="*70)
print("ORIGINAL DATASET")
print("="*70)

print(f"Rows    : {len(df)}")
print(f"Columns : {len(df.columns)}")

print("\nOriginal Columns")
print(df.columns.tolist())

# ============================================================
# STEP 3 : Standardize Column Names
# ============================================================

df.rename(columns={
    "Date":"date",
    "Open":"open_price",
    "High":"high_price",
    "Low":"low_price",
    "Close":"close_price",
    "Volume":"volume",
    "Dividends":"dividends",
    "Stock Splits":"stock_splits"
}, inplace=True)

# ============================================================
# STEP 4 : Remove Unnecessary Columns
# ============================================================

drop_cols = []

if "dividends" in df.columns:
    drop_cols.append("dividends")

if "stock_splits" in df.columns:
    drop_cols.append("stock_splits")

df.drop(columns=drop_cols, inplace=True)

# ============================================================
# STEP 5 : Clean Date
# ============================================================

df["date"] = pd.to_datetime(df["date"], utc=True)

# Remove timezone
df["date"] = df["date"].dt.tz_convert(None)

# Keep only Date
df["date"] = df["date"].dt.date

# Convert back to datetime
df["date"] = pd.to_datetime(df["date"])

# ============================================================
# STEP 6 : Create Calendar Columns
# ============================================================

df["year"] = df["date"].dt.year

df["month"] = df["date"].dt.month_name()

df["month_no"] = df["date"].dt.month

df["quarter"] = "Q" + df["date"].dt.quarter.astype(str)

df["week"] = df["date"].dt.isocalendar().week.astype(int)

df["day"] = df["date"].dt.day

df["day_name"] = df["date"].dt.day_name()

# ============================================================
# STEP 7 : Numeric Conversion
# ============================================================

numeric_cols = [
    "open_price",
    "high_price",
    "low_price",
    "close_price",
    "volume"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# ============================================================
# STEP 8 : Derived Business Metrics
# ============================================================

# Daily Price Change
df["price_change"] = (
    df["close_price"] - df["open_price"]
).round(2)

# Intraday Range
df["price_range"] = (
    df["high_price"] - df["low_price"]
).round(2)

# Daily Return %
df["daily_return_pct"] = (
    df["close_price"].pct_change() * 100
).round(2)

# 7-Day Moving Average
df["ma_7"] = (
    df["close_price"]
    .rolling(7)
    .mean()
    .round(2)
)

# 30-Day Moving Average
df["ma_30"] = (
    df["close_price"]
    .rolling(30)
    .mean()
    .round(2)
)

# ============================================================
# STEP 9 : Missing Values
# ============================================================

print("\nMissing Values")
print(df.isnull().sum())

# ============================================================
# STEP 10 : Remove Duplicates
# ============================================================

duplicates = df.duplicated().sum()

print("\nDuplicate Rows :", duplicates)

df.drop_duplicates(inplace=True)

# ============================================================
# STEP 11 : Sort Dataset
# ============================================================

df.sort_values("date", inplace=True)

df.reset_index(drop=True, inplace=True)

# ============================================================
# STEP 12 : Final Column Order
# ============================================================

df = df[[
    "date",

    "year",
    "month",
    "month_no",
    "quarter",
    "week",
    "day",
    "day_name",

    "open_price",
    "high_price",
    "low_price",
    "close_price",

    "price_change",
    "price_range",
    "daily_return_pct",

    "ma_7",
    "ma_30",

    "volume"
]]

# ============================================================
# STEP 13 : Data Quality Report
# ============================================================

print("\n")
print("="*70)
print("DATA QUALITY REPORT")
print("="*70)

print(f"\nRows : {len(df)}")

print(f"\nColumns : {len(df.columns)}")

print("\nData Types")
print(df.dtypes)

print("\nMissing Values")
print(df.isnull().sum())

print("\nDuplicate Rows")
print(df.duplicated().sum())

print("\nDate Range")
print(df["date"].min(), "to", df["date"].max())

print("\nSummary Statistics")
print(df.describe())

# ============================================================
# STEP 14 : Export Gold Dataset
# ============================================================

output = "Gold_DS_OIL_01_Brent_Crude.csv"

df.to_csv(output, index=False)

print("\nGold Dataset Created Successfully!")

# ============================================================
# STEP 15 : Download
# ============================================================

files.download(output)

Saving DS_OIL_01_Brent_Crude.csv to DS_OIL_01_Brent_Crude.csv
ORIGINAL DATASET
Rows    : 632
Columns : 8

Original Columns
['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Dividends', 'Stock Splits']

Missing Values
date                 0
open_price           0
high_price           0
low_price            0
close_price          0
volume               0
year                 0
month                0
month_no             0
quarter              0
week                 0
day                  0
day_name             0
price_change         0
price_range          0
daily_return_pct     1
ma_7                 6
ma_30               29
dtype: int64

Duplicate Rows : 0


DATA QUALITY REPORT

Rows : 632

Columns : 18

Data Types
date                datetime64[ns]
year                         int32
month                       object
month_no                     int32
quarter                     object
week                         int64
day                          int32
day_name                    o

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# Hormuz Impact Project
# ETL Module : Oil
# Dataset    : DS-OIL-02.csv
# Output     : Gold_DS_OIL_02_Crude_Oil_Imports.csv
# Version    : 2.1
# ============================================================

import pandas as pd
from google.colab import files

# ============================================================
# STEP 1 : Upload Dataset
# ============================================================

uploaded = files.upload()
filename = list(uploaded.keys())[0]

# ============================================================
# STEP 2 : Read Dataset
# ============================================================

df = pd.read_csv(filename)

print("="*70)
print("ORIGINAL DATASET")
print("="*70)

print(f"Rows    : {len(df)}")
print(f"Columns : {len(df.columns)}")

print("\nOriginal Columns")
print(df.columns.tolist())

# ============================================================
# STEP 3 : Filter Required Records
# ============================================================

df = df[
    (df["PRODUCTS"].str.upper().str.strip() == "CRUDE OIL") &
    (df["TRADE"].str.upper().str.strip() == "IMPORT")
].copy()

# ============================================================
# STEP 4 : Rename Columns
# ============================================================

df.rename(columns={
    "Month": "month",
    "Year": "year",
    "Quantity (000 Metric Tonnes)": "quantity_000_mt",
    "Value in Rupees (Crore)": "value_inr_crore",
    "Value in Dollars (Million US dollar)": "value_usd_million"
}, inplace=True)

# ============================================================
# STEP 5 : Month Mapping
# ============================================================

month_map = {
    "January": 1,
    "February": 2,
    "March": 3,
    "April": 4,
    "May": 5,
    "June": 6,
    "July": 7,
    "August": 8,
    "September": 9,
    "October": 10,
    "November": 11,
    "December": 12
}

df["month_no"] = df["month"].map(month_map)

# ============================================================
# STEP 6 : Date Columns
# ============================================================

df["date"] = pd.to_datetime(
    dict(
        year=df["year"],
        month=df["month_no"],
        day=1
    )
)

df["quarter"] = "Q" + df["date"].dt.quarter.astype(str)

# ============================================================
# STEP 7 : Numeric Conversion
# ============================================================

numeric_cols = [
    "quantity_000_mt",
    "value_inr_crore",
    "value_usd_million"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# ============================================================
# STEP 8 : Derived Business Metrics
# ============================================================

# Average Import Price (USD / MT)

df["avg_import_price_usd_per_mt"] = (
    (df["value_usd_million"] * 1000) /
    df["quantity_000_mt"]
).round(2)

# Average Import Price (INR / MT)

df["avg_import_price_inr_per_mt"] = (
    (df["value_inr_crore"] * 10000000) /
    (df["quantity_000_mt"] * 1000)
).round(2)

# ============================================================
# STEP 9 : Remove Unnecessary Columns
# ============================================================

drop_cols = [
    "PRODUCTS",
    "TRADE",
    "date_updated"
]

for col in drop_cols:
    if col in df.columns:
        df.drop(columns=col, inplace=True)

# ============================================================
# STEP 10 : Sort & Remove Duplicates
# ============================================================

df.drop_duplicates(inplace=True)

df.sort_values("date", inplace=True)

df.reset_index(drop=True, inplace=True)

# ============================================================
# STEP 11 : Final Column Order
# ============================================================

df = df[[
    "date",
    "year",
    "month",
    "month_no",
    "quarter",
    "quantity_000_mt",
    "value_inr_crore",
    "value_usd_million",
    "avg_import_price_usd_per_mt",
    "avg_import_price_inr_per_mt"
]]

# ============================================================
# STEP 12 : Data Quality Report
# ============================================================

print("\n")
print("="*70)
print("DATA QUALITY REPORT")
print("="*70)

print(f"\nRows : {len(df)}")
print(f"Columns : {len(df.columns)}")

print("\nMissing Values")
print(df.isnull().sum())

print("\nDuplicate Rows")
print(df.duplicated().sum())

print("\nDate Range")
print(df["date"].min(), "to", df["date"].max())

print("\nSummary Statistics")
print(df.describe())

# ============================================================
# STEP 13 : Export
# ============================================================

output = "Gold_DS_OIL_02_Crude_Oil_Imports.csv"

df.to_csv(output, index=False)

print("\nGold Dataset Created Successfully!")

# ============================================================
# STEP 14 : Download
# ============================================================

files.download(output)

Saving DS-OIL-02.csv to DS-OIL-02 (1).csv
ORIGINAL DATASET
Rows    : 312
Columns : 8

Original Columns
['Month', 'Year', 'PRODUCTS', 'TRADE', 'Quantity (000 Metric Tonnes)', 'Value in Rupees (Crore)', 'Value in Dollars (Million US dollar)', 'date_updated']


DATA QUALITY REPORT

Rows : 12
Columns : 10

Missing Values
date                           0
year                           0
month                          0
month_no                       0
quarter                        0
quantity_000_mt                0
value_inr_crore                0
value_usd_million              0
avg_import_price_usd_per_mt    0
avg_import_price_inr_per_mt    0
dtype: int64

Duplicate Rows
0

Date Range
2025-04-01 00:00:00 to 2026-03-01 00:00:00

Summary Statistics
                      date         year   month_no  quantity_000_mt  \
count                   12    12.000000  12.000000        12.000000   
mean   2025-09-15 18:00:00  2025.250000   6.500000     20480.722500   
min    2025-04-01 00:00:00  2025

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# Hormuz Impact Project
# ETL Module : Oil
# Dataset    : DS-OIL-03 Daily Retail Selling Price
# Output     : Gold_DS_OIL_03_Petrol_Diesel_Prices.csv
# Version    : 2.0
# ============================================================

import pandas as pd
from google.colab import files

# ============================================================
# STEP 1 : Upload
# ============================================================

uploaded = files.upload()
filename = list(uploaded.keys())[0]

# ============================================================
# STEP 2 : Read
# ============================================================

df = pd.read_csv(filename)

print("="*70)
print("ORIGINAL DATASET")
print("="*70)

print(f"Rows : {len(df)}")
print(f"Columns : {len(df.columns)}")

# ============================================================
# STEP 3 : Rename Columns
# ============================================================

df.rename(columns={
    "petrol_price_inr_per_litre":"petrol_price",
    "diesel_price_inr_per_litre":"diesel_price",
    "state_vat_petrol_pct":"petrol_vat_pct",
    "state_vat_diesel_pct":"diesel_vat_pct"
}, inplace=True)

# ============================================================
# STEP 4 : Date
# ============================================================

df["date"] = pd.to_datetime(df["date"])

df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month_name()
df["month_no"] = df["date"].dt.month
df["quarter"] = "Q" + df["date"].dt.quarter.astype(str)
df["week"] = df["date"].dt.isocalendar().week.astype(int)
df["day"] = df["date"].dt.day
df["day_name"] = df["date"].dt.day_name()

# ============================================================
# STEP 5 : Numeric Conversion
# ============================================================

numeric_cols = [
    "petrol_price",
    "diesel_price",
    "petrol_vat_pct",
    "diesel_vat_pct"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# ============================================================
# STEP 6 : Sort
# ============================================================

df.sort_values(["city","date"], inplace=True)

# ============================================================
# STEP 7 : Derived Business Metrics
# ============================================================

df["petrol_price_change"] = (
    df.groupby("city")["petrol_price"]
      .diff()
      .round(2)
)

df["diesel_price_change"] = (
    df.groupby("city")["diesel_price"]
      .diff()
      .round(2)
)

# ============================================================
# STEP 8 : Remove Duplicates
# ============================================================

df.drop_duplicates(inplace=True)

# ============================================================
# STEP 9 : Final Order
# ============================================================

df = df[
[
"date",
"year",
"month",
"month_no",
"quarter",
"week",
"day",
"day_name",
"city",
"state",
"petrol_price",
"diesel_price",
"petrol_vat_pct",
"diesel_vat_pct",
"petrol_price_change",
"diesel_price_change",
"is_long_freeze"
]
]

# ============================================================
# STEP 10 : Validation
# ============================================================

print("\n")
print("="*70)
print("DATA QUALITY REPORT")
print("="*70)

print("\nRows :", len(df))
print("\nColumns :", len(df.columns))

print("\nMissing Values")
print(df.isnull().sum())

print("\nDuplicate Rows")
print(df.duplicated().sum())

print("\nDate Range")
print(df["date"].min(),"to",df["date"].max())

# ============================================================
# STEP 11 : Export
# ============================================================

output = "Gold_DS_OIL_03_Petrol_Diesel_Prices.csv"

df.to_csv(output,index=False)

print("\nGold Dataset Created Successfully!")

files.download(output)

Saving DS-OIL-03 Daily Retail Selling Price of Petrol and Diesel in Metro Cities.csv to DS-OIL-03 Daily Retail Selling Price of Petrol and Diesel in Metro Cities.csv
ORIGINAL DATASET
Rows : 11058
Columns : 8


DATA QUALITY REPORT

Rows : 11058

Columns : 17

Missing Values
date                   0
year                   0
month                  0
month_no               0
quarter                0
week                   0
day                    0
day_name               0
city                   0
state                  0
petrol_price           0
diesel_price           0
petrol_vat_pct         0
diesel_vat_pct         0
petrol_price_change    6
diesel_price_change    6
is_long_freeze         0
dtype: int64

Duplicate Rows
0

Date Range
2021-05-01 00:00:00 to 2026-05-17 00:00:00

Gold Dataset Created Successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# Hormuz Impact Project
# ETL Module : Oil
# Dataset    : DS_OIL_04_India_Domestic_Crude_Production.csv
# Output     : Gold_DS_OIL_04_Domestic_Crude_Production.csv
# Version    : 2.0
# ============================================================

import pandas as pd
from google.colab import files

# ============================================================
# STEP 1 : Upload
# ============================================================

uploaded = files.upload()
filename = list(uploaded.keys())[0]

# ============================================================
# STEP 2 : Read Dataset
# ============================================================

df = pd.read_csv(filename)

print("="*70)
print("ORIGINAL DATASET")
print("="*70)

print(f"Rows    : {len(df)}")
print(f"Columns : {len(df.columns)}")

# ============================================================
# STEP 3 : Rename Columns
# ============================================================

df.rename(columns={
    "year": "financial_year"
}, inplace=True)

# ============================================================
# STEP 4 : Create Year Column
# ============================================================

df["year"] = (
    df["financial_year"]
      .str[:4]
      .astype(int)
)

# ============================================================
# STEP 5 : Create Date Column
# ============================================================

df["date"] = pd.to_datetime(
    df["year"].astype(str) + "-04-01"
)

# ============================================================
# STEP 6 : Numeric Conversion
# ============================================================

numeric_cols = [
    "production_mmt",
    "imports_mmt",
    "consumption_mmt"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# ============================================================
# STEP 7 : Derived KPIs
# ============================================================

df["import_dependency_pct"] = (
    (df["imports_mmt"] / df["consumption_mmt"]) * 100
).round(2)

df["domestic_production_share_pct"] = (
    (df["production_mmt"] / df["consumption_mmt"]) * 100
).round(2)

df["import_to_production_ratio"] = (
    df["imports_mmt"] / df["production_mmt"]
).round(2)

# ============================================================
# STEP 8 : Sort
# ============================================================

df.sort_values("date", inplace=True)
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

# ============================================================
# STEP 9 : Final Column Order
# ============================================================

df = df[
[
    "date",
    "financial_year",
    "year",
    "production_mmt",
    "imports_mmt",
    "consumption_mmt",
    "import_dependency_pct",
    "domestic_production_share_pct",
    "import_to_production_ratio"
]
]

# ============================================================
# STEP 10 : Validation
# ============================================================

print("\n")
print("="*70)
print("DATA QUALITY REPORT")
print("="*70)

print("\nRows :", len(df))
print("\nColumns :", len(df.columns))

print("\nMissing Values")
print(df.isnull().sum())

print("\nDuplicate Rows")
print(df.duplicated().sum())

print("\nSummary Statistics")
print(df.describe())

# ============================================================
# STEP 11 : Export
# ============================================================

output = "Gold_DS_OIL_04_Domestic_Crude_Production.csv"

df.to_csv(output, index=False)

print("\nGold Dataset Created Successfully!")

files.download(output)

Saving DS_OIL_04_India_Domestic_Crude_Production.csv to DS_OIL_04_India_Domestic_Crude_Production.csv
ORIGINAL DATASET
Rows    : 10
Columns : 4


DATA QUALITY REPORT

Rows : 10

Columns : 9

Missing Values
date                             0
financial_year                   0
year                             0
production_mmt                   0
imports_mmt                      0
consumption_mmt                  0
import_dependency_pct            0
domestic_production_share_pct    0
import_to_production_ratio       0
dtype: int64

Duplicate Rows
0

Summary Statistics
                      date        year  production_mmt  imports_mmt  \
count                   10    10.00000       10.000000    10.000000   
mean   2020-09-30 07:12:00  2020.50000       30.530000   185.850000   
min    2016-04-01 00:00:00  2016.00000       28.000000   163.800000   
25%    2018-07-01 06:00:00  2018.25000       28.825000   171.100000   
50%    2020-09-30 12:00:00  2020.50000       29.550000   180.650000   
75

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# Hormuz Impact Project
# ETL Module : Oil
# Dataset    : DS_OIL_05_Country_Wise_Crude_Oil_Import_Share_2025.csv
# Output     : Gold_DS_OIL_05_Country_Wise_Import_Share.csv
# Version    : 2.0
# ============================================================

import pandas as pd
from google.colab import files

# ============================================================
# STEP 1 : Upload
# ============================================================

uploaded = files.upload()
filename = list(uploaded.keys())[0]

# ============================================================
# STEP 2 : Read Dataset
# ============================================================

df = pd.read_csv(filename)

print("=" * 70)
print("ORIGINAL DATASET")
print("=" * 70)

print(f"Rows    : {len(df)}")
print(f"Columns : {len(df.columns)}")

# ============================================================
# STEP 3 : Rename Columns
# ============================================================

df.rename(columns={
    "share_percent": "import_share_pct",
    "uses_strait_of_hormuz": "hormuz_dependency"
}, inplace=True)

# ============================================================
# STEP 4 : Add Analysis Year
# ============================================================

df["analysis_year"] = 2025

# ============================================================
# STEP 5 : Standardize Boolean
# ============================================================

df["opec_member"] = df["opec_member"].map({
    True: "Yes",
    False: "No"
})

df["hormuz_dependency"] = df["hormuz_dependency"].map({
    True: "High",
    False: "Low"
})

# ============================================================
# STEP 6 : Risk Score
# ============================================================

risk_map = {
    "High": 5,
    "Medium": 3,
    "Low": 1
}

df["risk_score"] = df["risk_level"].map(risk_map)

# ============================================================
# STEP 7 : Numeric Conversion
# ============================================================

numeric_cols = [
    "rank",
    "import_value_usd_billion",
    "import_share_pct",
    "risk_score"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# ============================================================
# STEP 8 : Sort
# ============================================================

df.sort_values("rank", inplace=True)

df.drop_duplicates(inplace=True)

df.reset_index(drop=True, inplace=True)

# ============================================================
# STEP 9 : Final Column Order
# ============================================================

df = df[
    [
        "analysis_year",
        "rank",
        "country",
        "import_value_usd_billion",
        "import_share_pct",
        "opec_member",
        "hormuz_dependency",
        "risk_level",
        "risk_score"
    ]
]

# ============================================================
# STEP 10 : Validation
# ============================================================

print("\n")
print("=" * 70)
print("DATA QUALITY REPORT")
print("=" * 70)

print("\nRows :", len(df))
print("\nColumns :", len(df.columns))

print("\nMissing Values")
print(df.isnull().sum())

print("\nDuplicate Rows")
print(df.duplicated().sum())

# ============================================================
# STEP 11 : Export
# ============================================================

output = "Gold_DS_OIL_05_Country_Wise_Import_Share.csv"

df.to_csv(output, index=False)

print("\nGold Dataset Created Successfully!")

files.download(output)

Saving DS_OIL_05_Country_Wise_Crude_Oil_Import_Share_2025.csv to DS_OIL_05_Country_Wise_Crude_Oil_Import_Share_2025.csv
ORIGINAL DATASET
Rows    : 15
Columns : 7


DATA QUALITY REPORT

Rows : 15

Columns : 9

Missing Values
analysis_year               0
rank                        0
country                     0
import_value_usd_billion    0
import_share_pct            0
opec_member                 0
hormuz_dependency           0
risk_level                  0
risk_score                  0
dtype: int64

Duplicate Rows
0

Gold Dataset Created Successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# Hormuz Impact Project
# ETL Module : LPG
# Dataset    : DS-LPG-01 — Domestic LPG Production.csv
# Output     : Gold_DS_LPG_01_Domestic_LPG_Production.csv
# Version    : 1.0
# ============================================================

import pandas as pd
from google.colab import files

# ============================================================
# STEP 1 : Upload
# ============================================================

uploaded = files.upload()
filename = list(uploaded.keys())[0]

# ============================================================
# STEP 2 : Read Dataset
# ============================================================

df = pd.read_csv(filename)

print("="*70)
print("ORIGINAL DATASET")
print("="*70)

print(f"Rows    : {len(df)}")
print(f"Columns : {len(df.columns)}")

# ============================================================
# STEP 3 : Rename Columns
# ============================================================

df.rename(columns={
    "Financial Year": "financial_year",
    "LPG Production (MMT)": "lpg_production_mmt"
}, inplace=True)

# ============================================================
# STEP 4 : Create Year
# ============================================================

df["year"] = (
    df["financial_year"]
      .str[:4]
      .astype(int)
)

# ============================================================
# STEP 5 : Create Date
# ============================================================

# Use first day of financial year (1 April)
df["date"] = pd.to_datetime(
    df["year"].astype(str) + "-04-01"
)

# ============================================================
# STEP 6 : Numeric Conversion
# ============================================================

df["lpg_production_mmt"] = pd.to_numeric(
    df["lpg_production_mmt"],
    errors="coerce"
)

# ============================================================
# STEP 7 : Remove Duplicates
# ============================================================

df.drop_duplicates(inplace=True)

# ============================================================
# STEP 8 : Sort
# ============================================================

df.sort_values("date", inplace=True)
df.reset_index(drop=True, inplace=True)

# ============================================================
# STEP 9 : Final Column Order
# ============================================================

df = df[
    [
        "date",
        "financial_year",
        "year",
        "lpg_production_mmt"
    ]
]

# ============================================================
# STEP 10 : Validation
# ============================================================

print("\n")
print("="*70)
print("DATA QUALITY REPORT")
print("="*70)

print("\nRows :", len(df))
print("\nColumns :", len(df.columns))

print("\nMissing Values")
print(df.isnull().sum())

print("\nDuplicate Rows")
print(df.duplicated().sum())

print("\nSummary Statistics")
print(df.describe())

# ============================================================
# STEP 11 : Export
# ============================================================

output = "Gold_DS_LPG_01_Domestic_LPG_Production.csv"

df.to_csv(output, index=False)

print("\nGold Dataset Created Successfully!")

# ============================================================
# STEP 12 : Download
# ============================================================

files.download(output)

Saving DS-LPG-01 — Domestic LPG Production.csv to DS-LPG-01 — Domestic LPG Production.csv
ORIGINAL DATASET
Rows    : 2
Columns : 2


DATA QUALITY REPORT

Rows : 2

Columns : 4

Missing Values
date                  0
financial_year        0
year                  0
lpg_production_mmt    0
dtype: int64

Duplicate Rows
0

Summary Statistics
                      date         year  lpg_production_mmt
count                    2     2.000000            2.000000
mean   2024-09-30 12:00:00  2024.500000           12.950000
min    2024-04-01 00:00:00  2024.000000           12.800000
25%    2024-07-01 06:00:00  2024.250000           12.875000
50%    2024-09-30 12:00:00  2024.500000           12.950000
75%    2024-12-30 18:00:00  2024.750000           13.025000
max    2025-04-01 00:00:00  2025.000000           13.100000
std                    NaN     0.707107            0.212132

Gold Dataset Created Successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# Hormuz Impact Project
# ETL Module : LPG
# Dataset    : DS-LPG-02.csv
# Output     : Gold_DS_LPG_02_LPG_Imports.csv
# Version    : 1.0
# ============================================================

import pandas as pd
from google.colab import files

# ============================================================
# STEP 1 : Upload
# ============================================================

uploaded = files.upload()
filename = list(uploaded.keys())[0]

# ============================================================
# STEP 2 : Read Dataset
# ============================================================

df = pd.read_csv(filename)

print("=" * 70)
print("ORIGINAL DATASET")
print("=" * 70)

print(f"Rows    : {len(df)}")
print(f"Columns : {len(df.columns)}")

# ============================================================
# STEP 3 : Standardize Column Names
# ============================================================

df.rename(columns={
    "financial_year": "financial_year",
    "lpg_imports_mmt": "lpg_imports_mmt"
}, inplace=True)

# ============================================================
# STEP 4 : Create Year
# ============================================================

df["year"] = (
    df["financial_year"]
      .str[:4]
      .astype(int)
)

# ============================================================
# STEP 5 : Create Date
# ============================================================

df["date"] = pd.to_datetime(
    df["year"].astype(str) + "-04-01"
)

# ============================================================
# STEP 6 : Numeric Conversion
# ============================================================

df["lpg_imports_mmt"] = pd.to_numeric(
    df["lpg_imports_mmt"],
    errors="coerce"
)

# ============================================================
# STEP 7 : Remove Duplicates
# ============================================================

df.drop_duplicates(inplace=True)

# ============================================================
# STEP 8 : Sort
# ============================================================

df.sort_values("date", inplace=True)
df.reset_index(drop=True, inplace=True)

# ============================================================
# STEP 9 : Final Column Order
# ============================================================

df = df[
    [
        "date",
        "financial_year",
        "year",
        "lpg_imports_mmt"
    ]
]

# ============================================================
# STEP 10 : Validation
# ============================================================

print("\n" + "=" * 70)
print("DATA QUALITY REPORT")
print("=" * 70)

print("\nRows :", len(df))
print("\nColumns :", len(df.columns))

print("\nMissing Values")
print(df.isnull().sum())

print("\nDuplicate Rows")
print(df.duplicated().sum())

print("\nSummary Statistics")
print(df.describe())

# ============================================================
# STEP 11 : Export
# ============================================================

output = "Gold_DS_LPG_02_LPG_Imports.csv"

df.to_csv(output, index=False)

print("\nGold Dataset Created Successfully!")

# ============================================================
# STEP 12 : Download
# ============================================================

files.download(output)

Saving DS-LPG-02.csv to DS-LPG-02.csv
ORIGINAL DATASET
Rows    : 1
Columns : 2

DATA QUALITY REPORT

Rows : 1

Columns : 4

Missing Values
date               0
financial_year     0
year               0
lpg_imports_mmt    0
dtype: int64

Duplicate Rows
0

Summary Statistics
                      date    year  lpg_imports_mmt
count                    1     1.0             1.00
mean   2024-04-01 00:00:00  2024.0            20.67
min    2024-04-01 00:00:00  2024.0            20.67
25%    2024-04-01 00:00:00  2024.0            20.67
50%    2024-04-01 00:00:00  2024.0            20.67
75%    2024-04-01 00:00:00  2024.0            20.67
max    2024-04-01 00:00:00  2024.0            20.67
std                    NaN     NaN              NaN

Gold Dataset Created Successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# Hormuz Impact Project
# ETL Module : LPG
# Dataset    : DS-LPG-03.csv
# Output     : Gold_DS_LPG_03_LPG_Consumption.csv
# Version    : 1.0
# ============================================================

import pandas as pd
from google.colab import files

# ============================================================
# STEP 1 : Upload
# ============================================================

uploaded = files.upload()
filename = list(uploaded.keys())[0]

# ============================================================
# STEP 2 : Read Dataset
# ============================================================

df = pd.read_csv(filename)

print("=" * 70)
print("ORIGINAL DATASET")
print("=" * 70)

print(f"Rows    : {len(df)}")
print(f"Columns : {len(df.columns)}")

# ============================================================
# STEP 3 : Clean Financial Year
# ============================================================

# Remove (P), (RE), etc.
df["financial_year"] = (
    df["financial_year"]
    .astype(str)
    .str.replace(r"\s*\(.*?\)", "", regex=True)
    .str.strip()
)

# ============================================================
# STEP 4 : Create Year
# ============================================================

df["year"] = (
    df["financial_year"]
      .str[:4]
      .astype(int)
)

# ============================================================
# STEP 5 : Create Date
# ============================================================

df["date"] = pd.to_datetime(
    df["year"].astype(str) + "-04-01"
)

# ============================================================
# STEP 6 : Numeric Conversion
# ============================================================

numeric_cols = [
    "lpg_consumption_000_metric_tonnes",
    "lpg_consumption_mmt"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# ============================================================
# STEP 7 : Remove Duplicates
# ============================================================

df.drop_duplicates(inplace=True)

# ============================================================
# STEP 8 : Sort
# ============================================================

df.sort_values("date", inplace=True)
df.reset_index(drop=True, inplace=True)

# ============================================================
# STEP 9 : Final Column Order
# ============================================================

df = df[
    [
        "date",
        "financial_year",
        "year",
        "lpg_consumption_000_metric_tonnes",
        "lpg_consumption_mmt"
    ]
]

# ============================================================
# STEP 10 : Validation
# ============================================================

print("\n" + "=" * 70)
print("DATA QUALITY REPORT")
print("=" * 70)

print("\nRows :", len(df))
print("\nColumns :", len(df.columns))

print("\nMissing Values")
print(df.isnull().sum())

print("\nDuplicate Rows")
print(df.duplicated().sum())

print("\nSummary Statistics")
print(df.describe())

# ============================================================
# STEP 11 : Export
# ============================================================

output = "Gold_DS_LPG_03_LPG_Consumption.csv"

df.to_csv(output, index=False)

print("\nGold Dataset Created Successfully!")

# ============================================================
# STEP 12 : Download
# ============================================================

files.download(output)

Saving DS-LPG-03.csv to DS-LPG-03.csv
ORIGINAL DATASET
Rows    : 4
Columns : 3

DATA QUALITY REPORT

Rows : 4

Columns : 5

Missing Values
date                                 0
financial_year                       0
year                                 0
lpg_consumption_000_metric_tonnes    0
lpg_consumption_mmt                  0
dtype: int64

Duplicate Rows
0

Summary Statistics
                      date         year  lpg_consumption_000_metric_tonnes  \
count                    4     4.000000                            4.00000   
mean   2023-10-01 00:00:00  2023.500000                        30676.84150   
min    2022-04-01 00:00:00  2022.000000                        28503.83200   
25%    2022-12-30 18:00:00  2022.750000                        29373.67000   
50%    2023-10-01 00:00:00  2023.500000                        30495.89700   
75%    2024-07-01 06:00:00  2024.250000                        31799.06850   
max    2025-04-01 00:00:00  2025.000000                        33211.

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# Hormuz Impact Project
# ETL Module : LPG
# Dataset    : DS_LPG_04_Monthly_Cylinder_Prices_2024_2026.xlsx
# Output     : Gold_DS_LPG_04_Monthly_Cylinder_Prices.csv
# Version    : 2.0
# ============================================================

import pandas as pd
from google.colab import files

# ============================================================
# Upload
# ============================================================

uploaded = files.upload()
filename = list(uploaded.keys())[0]

# ============================================================
# Read Excel
# ============================================================

df = pd.read_excel(filename)

print("="*70)
print("ORIGINAL DATASET")
print("="*70)

print(df.head())

print(df.columns)

# ============================================================
# Rename Columns
# ============================================================

df.rename(columns={
    df.columns[0]: "date",
    df.columns[1]: "domestic_lpg_price"
}, inplace=True)

# ============================================================
# Date
# ============================================================

df["date"] = pd.to_datetime(df["date"])

df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month_name()
df["month_no"] = df["date"].dt.month
df["quarter"] = "Q" + df["date"].dt.quarter.astype(str)

# ============================================================
# Numeric
# ============================================================

df["domestic_lpg_price"] = pd.to_numeric(
    df["domestic_lpg_price"],
    errors="coerce"
)

# ============================================================
# Derived Metrics
# ============================================================

df["monthly_price_change"] = (
    df["domestic_lpg_price"]
      .diff()
      .round(2)
)

df["monthly_price_change_pct"] = (
    df["domestic_lpg_price"]
      .pct_change() * 100
).round(2)

# ============================================================
# Remove Duplicates
# ============================================================

df.drop_duplicates(inplace=True)

# ============================================================
# Sort
# ============================================================

df.sort_values("date", inplace=True)

df.reset_index(drop=True, inplace=True)

# ============================================================
# Final Columns
# ============================================================

df = df[
[
"date",
"year",
"month",
"month_no",
"quarter",
"domestic_lpg_price",
"monthly_price_change",
"monthly_price_change_pct"
]
]

# ============================================================
# Validation
# ============================================================

print("="*70)
print("DATA QUALITY REPORT")
print("="*70)

print(df.isnull().sum())

print(df.describe())

# ============================================================
# Export
# ============================================================

output="Gold_DS_LPG_04_Monthly_Cylinder_Prices.csv"

df.to_csv(output,index=False)

print("Gold Dataset Created Successfully!")

files.download(output)

Saving DS_LPG_04_Monthly_Cylinder_Prices_2024_2026.xlsx to DS_LPG_04_Monthly_Cylinder_Prices_2024_2026.xlsx
ORIGINAL DATASET
     month financial_year  delhi_domestic_lpg_14_2kg_price_rs
0  2024-04        2024-25                                 803
1  2024-05        2024-25                                 803
2  2024-06        2024-25                                 803
3  2024-07        2024-25                                 803
4  2024-08        2024-25                                 803
Index(['month', 'financial_year', 'delhi_domestic_lpg_14_2kg_price_rs'], dtype='object')
DATA QUALITY REPORT
date                         0
year                         0
month                        0
month_no                     0
quarter                      0
domestic_lpg_price          24
monthly_price_change        24
monthly_price_change_pct    24
dtype: int64
                      date         year   month_no  domestic_lpg_price  \
count                   24    24.000000  24.000000         

/tmp/ipykernel_2337/4287234286.py:74: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  .pct_change() * 100


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# Hormuz Impact Project
# ETL Module : LPG
# Dataset    : DS-LPG-05 — State-wise Active Domestic LPG Connections.xlsx
# Output     : Gold_DS_LPG_05_State_Wise_LPG_Connections.csv
# Version    : 2.0
# ============================================================

import pandas as pd
from google.colab import files

# ============================================================
# STEP 1 : Upload
# ============================================================

uploaded = files.upload()
filename = list(uploaded.keys())[0]

# ============================================================
# STEP 2 : Read Excel
# ============================================================

df = pd.read_excel(
    filename,
    sheet_name=0,
    skiprows=3
)

print("=" * 70)
print("ORIGINAL DATASET")
print("=" * 70)

print(df.head())

# ============================================================
# STEP 3 : Rename Columns
# ============================================================

df.columns = [
    "state",
    "analysis_date"
]

# ============================================================
# STEP 4 : Rename Value Column
# ============================================================

df.rename(columns={
    "analysis_date": "active_lpg_connections_lakh"
}, inplace=True)

# Since the second column header is actually a date, capture it
analysis_date = pd.Timestamp("2025-04-01")

df["analysis_date"] = analysis_date
df["analysis_year"] = analysis_date.year

# ============================================================
# STEP 5 : Clean State Names
# ============================================================

df["state"] = (
    df["state"]
      .astype(str)
      .str.strip()
      .str.title()
)

# ============================================================
# STEP 6 : Numeric Conversion
# ============================================================

df["active_lpg_connections_lakh"] = pd.to_numeric(
    df["active_lpg_connections_lakh"],
    errors="coerce"
)

# ============================================================
# STEP 7 : Derived Column
# ============================================================

df["active_lpg_connections"] = (
    df["active_lpg_connections_lakh"] * 100000
).round().astype("Int64")

# ============================================================
# STEP 8 : Remove Duplicates
# ============================================================

df.drop_duplicates(inplace=True)

# ============================================================
# STEP 9 : Sort
# ============================================================

df.sort_values("state", inplace=True)
df.reset_index(drop=True, inplace=True)

# ============================================================
# STEP 10 : Final Column Order
# ============================================================

df = df[
    [
        "analysis_date",
        "analysis_year",
        "state",
        "active_lpg_connections_lakh",
        "active_lpg_connections"
    ]
]

# ============================================================
# STEP 11 : Validation
# ============================================================

print("\n" + "=" * 70)
print("DATA QUALITY REPORT")
print("=" * 70)

print(df.info())

print("\nMissing Values")
print(df.isnull().sum())

print("\nDuplicate Rows")
print(df.duplicated().sum())

# ============================================================
# STEP 12 : Export
# ============================================================

output = "Gold_DS_LPG_05_State_Wise_LPG_Connections.csv"

df.to_csv(output, index=False)

print("\nGold Dataset Created Successfully!")

files.download(output)

Saving DS-LPG-05 — State-wise Active Domestic LPG Connections.xlsx to DS-LPG-05 — State-wise Active Domestic LPG Connections.xlsx
ORIGINAL DATASET
           STATE/UT  2025-04-01 00:00:00
0        CHANDIGARH              3.06663
1             DELHI             56.89812
2           HARYANA             82.60045
3  HIMACHAL PRADESH             22.43595
4   JAMMU & KASHMIR             35.42130

DATA QUALITY REPORT
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43 entries, 0 to 42
Data columns (total 5 columns):
 #   Column                       Non-Null Count  Dtype        
---  ------                       --------------  -----        
 0   analysis_date                43 non-null     datetime64[s]
 1   analysis_year                43 non-null     int64        
 2   state                        43 non-null     object       
 3   active_lpg_connections_lakh  42 non-null     float64      
 4   active_lpg_connections       42 non-null     Int64        
dtypes: Int64(1), datetime64[s](1), 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# Hormuz Impact Project
# ETL Module : LPG
# Dataset    : Gold_DS_LPG_06_LPG_Supply_Analysis.csv
# Version    : 1.0
# ============================================================

import pandas as pd
from google.colab import files

print("Upload these files:")
print("1. Gold_DS_LPG_01_Domestic_LPG_Production.csv")
print("2. Gold_DS_LPG_02_LPG_Imports.csv")
print("3. Gold_DS_LPG_03_LPG_Consumption.csv")

uploaded = files.upload()

production = None
imports = None
consumption = None

for file in uploaded.keys():

    if "LPG_01" in file:
        production = pd.read_csv(file)

    elif "LPG_02" in file:
        imports = pd.read_csv(file)

    elif "LPG_03" in file:
        consumption = pd.read_csv(file)

# Merge datasets
df = production.merge(
    imports,
    on=["date", "financial_year", "year"],
    how="outer"
)

df = df.merge(
    consumption,
    on=["date", "financial_year", "year"],
    how="outer"
)

# Numeric conversion
numeric_cols = [
    "lpg_production_mmt",
    "lpg_imports_mmt",
    "lpg_consumption_mmt"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# ============================================================
# Business KPIs
# ============================================================

df["import_dependency_pct"] = (
    df["lpg_imports_mmt"] /
    df["lpg_consumption_mmt"] * 100
).round(2)

df["domestic_production_share_pct"] = (
    df["lpg_production_mmt"] /
    df["lpg_consumption_mmt"] * 100
).round(2)

df["supply_gap_mmt"] = (
    df["lpg_consumption_mmt"] -
    df["lpg_production_mmt"]
).round(2)

df["import_to_production_ratio"] = (
    df["lpg_imports_mmt"] /
    df["lpg_production_mmt"]
).round(2)

df["net_supply_mmt"] = (
    df["lpg_production_mmt"] +
    df["lpg_imports_mmt"]
).round(2)

df["supply_surplus_deficit_mmt"] = (
    df["net_supply_mmt"] -
    df["lpg_consumption_mmt"]
).round(2)

# Sort
df.sort_values("date", inplace=True)

# Final column order
df = df[
[
    "date",
    "financial_year",
    "year",
    "lpg_production_mmt",
    "lpg_imports_mmt",
    "lpg_consumption_mmt",
    "import_dependency_pct",
    "domestic_production_share_pct",
    "supply_gap_mmt",
    "import_to_production_ratio",
    "net_supply_mmt",
    "supply_surplus_deficit_mmt"
]
]

# Validation
print("\n==============================")
print("DATA QUALITY REPORT")
print("==============================")

print(df.info())

print("\nMissing Values")
print(df.isnull().sum())

print("\nDuplicate Rows")
print(df.duplicated().sum())

# Export
output = "Gold_DS_LPG_06_LPG_Supply_Analysis.csv"

df.to_csv(output, index=False)

print("\nGold Dataset Created Successfully!")

files.download(output)

Upload these files:
1. Gold_DS_LPG_01_Domestic_LPG_Production.csv
2. Gold_DS_LPG_02_LPG_Imports.csv
3. Gold_DS_LPG_03_LPG_Consumption.csv


Saving Gold_DS_LPG_03_LPG_Consumption.csv to Gold_DS_LPG_03_LPG_Consumption (1).csv
Saving Gold_DS_LPG_02_LPG_Imports.csv to Gold_DS_LPG_02_LPG_Imports (1).csv
Saving Gold_DS_LPG_01_Domestic_LPG_Production.csv to Gold_DS_LPG_01_Domestic_LPG_Production (2).csv

DATA QUALITY REPORT
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 12 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   date                           4 non-null      object 
 1   financial_year                 4 non-null      object 
 2   year                           4 non-null      int64  
 3   lpg_production_mmt             2 non-null      float64
 4   lpg_imports_mmt                1 non-null      float64
 5   lpg_consumption_mmt            4 non-null      float64
 6   import_dependency_pct          1 non-null      float64
 7   domestic_production_share_pct  2 non-null      float64
 8   supply_gap_mm

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# Hormuz Impact Project
# ETL Module : Fertilizer
# Dataset    : DS-FERT-01_Fertilizer_Production.csv
# Output     : Gold_DS_FERT_01_Fertilizer_Production.csv
# Version    : 1.0
# ============================================================

import pandas as pd
from google.colab import files

# ============================================================
# STEP 1 : Upload
# ============================================================

uploaded = files.upload()
filename = list(uploaded.keys())[0]

# ============================================================
# STEP 2 : Read Dataset
# ============================================================

df = pd.read_csv(filename)

print("="*70)
print("ORIGINAL DATASET")
print("="*70)

print(df.head())

# ============================================================
# STEP 3 : Clean Financial Year
# ============================================================

df["financial_year"] = (
    df["financial_year"]
      .astype(str)
      .str.replace(r"\s*\(.*?\)", "", regex=True)
      .str.strip()
)

# ============================================================
# STEP 4 : Create Year
# ============================================================

df["year"] = (
    df["financial_year"]
      .str[:4]
      .astype(int)
)

# ============================================================
# STEP 5 : Create Date
# ============================================================

df["date"] = pd.to_datetime(
    df["year"].astype(str) + "-04-01"
)

# ============================================================
# STEP 6 : Numeric Conversion
# ============================================================

numeric_cols = [
    "urea_production_lakh_mt",
    "dap_production_lakh_mt"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# ============================================================
# STEP 7 : Derived KPI
# ============================================================

df["total_production_lakh_mt"] = (
    df["urea_production_lakh_mt"] +
    df["dap_production_lakh_mt"]
).round(2)

# ============================================================
# STEP 8 : Remove Duplicates
# ============================================================

df.drop_duplicates(inplace=True)

# ============================================================
# STEP 9 : Sort
# ============================================================

df.sort_values("date", inplace=True)
df.reset_index(drop=True, inplace=True)

# ============================================================
# STEP 10 : Final Column Order
# ============================================================

df = df[
[
    "date",
    "financial_year",
    "year",
    "urea_production_lakh_mt",
    "dap_production_lakh_mt",
    "total_production_lakh_mt"
]
]

# ============================================================
# STEP 11 : Validation
# ============================================================

print("\n")
print("="*70)
print("DATA QUALITY REPORT")
print("="*70)

print(df.info())

print("\nMissing Values")
print(df.isnull().sum())

print("\nDuplicate Rows")
print(df.duplicated().sum())

print("\nSummary Statistics")
print(df.describe())

# ============================================================
# STEP 12 : Export
# ============================================================

output = "Gold_DS_FERT_01_Fertilizer_Production.csv"

df.to_csv(output, index=False)

print("\nGold Dataset Created Successfully!")

files.download(output)

Saving DS-FERT-01_Fertilizer_Production.csv to DS-FERT-01_Fertilizer_Production.csv
ORIGINAL DATASET
      financial_year  urea_production_lakh_mt  dap_production_lakh_mt
0            2024-25                   258.48                   34.25
1  2025-26 (Apr-Jan)                   251.26                   33.71


DATA QUALITY REPORT
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 6 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   date                      2 non-null      datetime64[ns]
 1   financial_year            2 non-null      object        
 2   year                      2 non-null      int64         
 3   urea_production_lakh_mt   2 non-null      float64       
 4   dap_production_lakh_mt    2 non-null      float64       
 5   total_production_lakh_mt  2 non-null      float64       
dtypes: datetime64[ns](1), float64(3), int64(1), object(1)
memory usage

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# Hormuz Impact Project
# ETL Module : Fertilizer
# Dataset    : DS-FERT-02_Fertilizer_Imports.csv
# Output     : Gold_DS_FERT_02_Fertilizer_Imports.csv
# Version    : 1.0
# ============================================================

import pandas as pd
from google.colab import files

# ============================================================
# STEP 1 : Upload
# ============================================================

uploaded = files.upload()
filename = list(uploaded.keys())[0]

# ============================================================
# STEP 2 : Read Dataset
# ============================================================

df = pd.read_csv(filename)

print("="*70)
print("ORIGINAL DATASET")
print("="*70)

print(df.head())

# ============================================================
# STEP 3 : Clean Financial Year
# ============================================================

df["financial_year"] = (
    df["financial_year"]
      .astype(str)
      .str.replace(r"\s*\(.*?\)", "", regex=True)
      .str.strip()
)

# ============================================================
# STEP 4 : Create Year
# ============================================================

df["year"] = (
    df["financial_year"]
      .str[:4]
      .astype(int)
)

# ============================================================
# STEP 5 : Create Date
# ============================================================

df["date"] = pd.to_datetime(
    df["year"].astype(str) + "-04-01"
)

# ============================================================
# STEP 6 : Numeric Conversion
# ============================================================

numeric_cols = [
    "urea_import_lakh_mt",
    "dap_import_lakh_mt"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# ============================================================
# STEP 7 : Derived KPI
# ============================================================

df["total_imports_lakh_mt"] = (
    df["urea_import_lakh_mt"] +
    df["dap_import_lakh_mt"]
).round(2)

# ============================================================
# STEP 8 : Remove Duplicates
# ============================================================

df.drop_duplicates(inplace=True)

# ============================================================
# STEP 9 : Sort
# ============================================================

df.sort_values("date", inplace=True)
df.reset_index(drop=True, inplace=True)

# ============================================================
# STEP 10 : Final Column Order
# ============================================================

df = df[
[
    "date",
    "financial_year",
    "year",
    "urea_import_lakh_mt",
    "dap_import_lakh_mt",
    "total_imports_lakh_mt"
]
]

# ============================================================
# STEP 11 : Validation
# ============================================================

print("\n")
print("="*70)
print("DATA QUALITY REPORT")
print("="*70)

print(df.info())

print("\nMissing Values")
print(df.isnull().sum())

print("\nDuplicate Rows")
print(df.duplicated().sum())

print("\nSummary Statistics")
print(df.describe())

# ============================================================
# STEP 12 : Export
# ============================================================

output = "Gold_DS_FERT_02_Fertilizer_Imports.csv"

df.to_csv(output, index=False)

print("\nGold Dataset Created Successfully!")

files.download(output)

Saving DS-FERT-02_Fertilizer_Imports.csv to DS-FERT-02_Fertilizer_Imports.csv
ORIGINAL DATASET
      financial_year  urea_import_lakh_mt  dap_import_lakh_mt
0            2024-25                 58.0               44.19
1  2025-26 (Apr-Sep)                 39.8               28.00


DATA QUALITY REPORT
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   date                   2 non-null      datetime64[ns]
 1   financial_year         2 non-null      object        
 2   year                   2 non-null      int64         
 3   urea_import_lakh_mt    2 non-null      float64       
 4   dap_import_lakh_mt     2 non-null      float64       
 5   total_imports_lakh_mt  2 non-null      float64       
dtypes: datetime64[ns](1), float64(3), int64(1), object(1)
memory usage: 228.0+ bytes
None

Missing Values
date              

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# Hormuz Impact Project
# ETL Module : Fertilizer
# Dataset    : DS-FERT-03_Fertilizer_Consumption.csv
# Output     : Gold_DS_FERT_03_Fertilizer_Consumption.csv
# Version    : 1.0
# ============================================================

import pandas as pd
from google.colab import files

# ============================================================
# STEP 1 : Upload
# ============================================================

uploaded = files.upload()
filename = list(uploaded.keys())[0]

# ============================================================
# STEP 2 : Read Dataset
# ============================================================

df = pd.read_csv(filename)

print("="*70)
print("ORIGINAL DATASET")
print("="*70)

print(df.head())

# ============================================================
# STEP 3 : Clean Financial Year
# ============================================================

df["financial_year"] = (
    df["financial_year"]
      .astype(str)
      .str.replace(r"\s*\(.*?\)", "", regex=True)
      .str.strip()
)

# ============================================================
# STEP 4 : Create Year
# ============================================================

df["year"] = (
    df["financial_year"]
      .str[:4]
      .astype(int)
)

# ============================================================
# STEP 5 : Create Date
# ============================================================

df["date"] = pd.to_datetime(
    df["year"].astype(str) + "-04-01"
)

# ============================================================
# STEP 6 : Numeric Conversion
# ============================================================

numeric_cols = [
    "total_fertilizer_consumption_million_mt",
    "urea_consumption_million_mt",
    "dap_consumption_million_mt"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# ============================================================
# STEP 7 : Remove Duplicates
# ============================================================

df.drop_duplicates(inplace=True)

# ============================================================
# STEP 8 : Sort
# ============================================================

df.sort_values("date", inplace=True)
df.reset_index(drop=True, inplace=True)

# ============================================================
# STEP 9 : Final Column Order
# ============================================================

df = df[
[
    "date",
    "financial_year",
    "year",
    "total_fertilizer_consumption_million_mt",
    "urea_consumption_million_mt",
    "dap_consumption_million_mt"
]
]

# ============================================================
# STEP 10 : Validation
# ============================================================

print("\n")
print("="*70)
print("DATA QUALITY REPORT")
print("="*70)

print(df.info())

print("\nMissing Values")
print(df.isnull().sum())

print("\nDuplicate Rows")
print(df.duplicated().sum())

print("\nSummary Statistics")
print(df.describe())

# ============================================================
# STEP 11 : Export
# ============================================================

output = "Gold_DS_FERT_03_Fertilizer_Consumption.csv"

df.to_csv(output, index=False)

print("\nGold Dataset Created Successfully!")

files.download(output)

Saving DS-FERT-03_Fertilizer_Consumption.csv to DS-FERT-03_Fertilizer_Consumption.csv
ORIGINAL DATASET
   financial_year  total_fertilizer_consumption_million_mt  \
0         2024-25                                     70.7   
1  2025-26 (Est.)                                     72.0   

   urea_consumption_million_mt  dap_consumption_million_mt  
0                        38.77                         9.4  
1                        39.50                         9.7  


DATA QUALITY REPORT
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 6 columns):
 #   Column                                   Non-Null Count  Dtype         
---  ------                                   --------------  -----         
 0   date                                     2 non-null      datetime64[ns]
 1   financial_year                           2 non-null      object        
 2   year                                     2 non-null      int64         
 3   total_fertilize

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# Hormuz Impact Project
# ADS Module : Fertilizer
# Dataset    : ADS_FERT_01_Fertilizer_Supply_Analysis.csv
# ============================================================

import pandas as pd
from google.colab import files

print("Upload the following files:")
print("1. Gold_DS_FERT_01_Fertilizer_Production.csv")
print("2. Gold_DS_FERT_02_Fertilizer_Imports.csv")
print("3. Gold_DS_FERT_03_Fertilizer_Consumption.csv")

uploaded = files.upload()

production = None
imports = None
consumption = None

for file in uploaded.keys():

    if "FERT_01" in file:
        production = pd.read_csv(file)

    elif "FERT_02" in file:
        imports = pd.read_csv(file)

    elif "FERT_03" in file:
        consumption = pd.read_csv(file)

# ============================================================
# Convert Consumption Unit
# Million MT -> Lakh MT
# ============================================================

consumption["total_consumption_lakh_mt"] = (
    consumption["total_fertilizer_consumption_million_mt"] * 10
).round(2)

consumption["urea_consumption_lakh_mt"] = (
    consumption["urea_consumption_million_mt"] * 10
).round(2)

consumption["dap_consumption_lakh_mt"] = (
    consumption["dap_consumption_million_mt"] * 10
).round(2)

# ============================================================
# Merge
# ============================================================

df = production.merge(
    imports,
    on=["date","financial_year","year"],
    how="outer"
)

df = df.merge(
    consumption[
        [
            "date",
            "financial_year",
            "year",
            "total_consumption_lakh_mt",
            "urea_consumption_lakh_mt",
            "dap_consumption_lakh_mt"
        ]
    ],
    on=["date","financial_year","year"],
    how="outer"
)

# ============================================================
# Business KPIs
# ============================================================

df["import_dependency_pct"] = (
    df["total_imports_lakh_mt"] /
    df["total_consumption_lakh_mt"] * 100
).round(2)

df["domestic_production_share_pct"] = (
    df["total_production_lakh_mt"] /
    df["total_consumption_lakh_mt"] * 100
).round(2)

df["supply_gap_lakh_mt"] = (
    df["total_consumption_lakh_mt"] -
    df["total_production_lakh_mt"]
).round(2)

df["import_to_production_ratio"] = (
    df["total_imports_lakh_mt"] /
    df["total_production_lakh_mt"]
).round(2)

df["net_supply_lakh_mt"] = (
    df["total_production_lakh_mt"] +
    df["total_imports_lakh_mt"]
).round(2)

df["supply_surplus_deficit_lakh_mt"] = (
    df["net_supply_lakh_mt"] -
    df["total_consumption_lakh_mt"]
).round(2)

# ============================================================
# Final Columns
# ============================================================

df = df[
[
    "date",
    "financial_year",
    "year",

    "total_production_lakh_mt",
    "total_imports_lakh_mt",
    "total_consumption_lakh_mt",

    "import_dependency_pct",
    "domestic_production_share_pct",
    "supply_gap_lakh_mt",
    "import_to_production_ratio",
    "net_supply_lakh_mt",
    "supply_surplus_deficit_lakh_mt"
]
]

# ============================================================
# Validation
# ============================================================

print("="*70)
print("DATA QUALITY REPORT")
print("="*70)

print(df.info())

print(df.describe())

print(df.isnull().sum())

# ============================================================
# Export
# ============================================================

output = "ADS_FERT_01_Fertilizer_Supply_Analysis.csv"

df.to_csv(output,index=False)

print("ADS Dataset Created Successfully!")

files.download(output)

Upload the following files:
1. Gold_DS_FERT_01_Fertilizer_Production.csv
2. Gold_DS_FERT_02_Fertilizer_Imports.csv
3. Gold_DS_FERT_03_Fertilizer_Consumption.csv


Saving Gold_DS_FERT_03_Fertilizer_Consumption.csv to Gold_DS_FERT_03_Fertilizer_Consumption (1).csv
Saving Gold_DS_FERT_02_Fertilizer_Imports.csv to Gold_DS_FERT_02_Fertilizer_Imports (1).csv
Saving Gold_DS_FERT_01_Fertilizer_Production.csv to Gold_DS_FERT_01_Fertilizer_Production (1).csv
DATA QUALITY REPORT
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 12 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   date                            2 non-null      object 
 1   financial_year                  2 non-null      object 
 2   year                            2 non-null      int64  
 3   total_production_lakh_mt        2 non-null      float64
 4   total_imports_lakh_mt           2 non-null      float64
 5   total_consumption_lakh_mt       2 non-null      float64
 6   import_dependency_pct           2 non-null      float64
 7   domestic_production_share_pct   2 n

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# Hormuz Impact Project
# ETL Module : Fertilizer
# Dataset    : DS-FERT-04_Fertilizer_Subsidy.csv
# Output     : Gold_DS_FERT_04_Fertilizer_Subsidy.csv
# Version    : 1.0
# ============================================================

import pandas as pd
from google.colab import files

# ============================================================
# STEP 1 : Upload Dataset
# ============================================================

uploaded = files.upload()
filename = list(uploaded.keys())[0]

# ============================================================
# STEP 2 : Read Dataset
# ============================================================

df = pd.read_csv(filename)

print("="*70)
print("ORIGINAL DATASET")
print("="*70)

print(df.head())

# ============================================================
# STEP 3 : Clean Financial Year
# ============================================================

df["financial_year"] = (
    df["financial_year"]
      .astype(str)
      .str.replace(r"\s*\(.*?\)", "", regex=True)
      .str.strip()
)

# ============================================================
# STEP 4 : Create Year
# ============================================================

df["year"] = (
    df["financial_year"]
      .str[:4]
      .astype(int)
)

# ============================================================
# STEP 5 : Create Date
# ============================================================

df["date"] = pd.to_datetime(
    df["year"].astype(str) + "-04-01"
)

# ============================================================
# STEP 6 : Numeric Conversion
# ============================================================

df["fertilizer_subsidy_rs_crore"] = pd.to_numeric(
    df["fertilizer_subsidy_rs_crore"],
    errors="coerce"
)

# ============================================================
# STEP 7 : Remove Duplicates
# ============================================================

df.drop_duplicates(inplace=True)

# ============================================================
# STEP 8 : Sort
# ============================================================

df.sort_values("date", inplace=True)
df.reset_index(drop=True, inplace=True)

# ============================================================
# STEP 9 : Final Column Order
# ============================================================

df = df[
    [
        "date",
        "financial_year",
        "year",
        "fertilizer_subsidy_rs_crore"
    ]
]

# ============================================================
# STEP 10 : Validation
# ============================================================

print("\n" + "="*70)
print("DATA QUALITY REPORT")
print("="*70)

print("\nRows :", len(df))
print("\nColumns :", len(df.columns))

print("\nMissing Values")
print(df.isnull().sum())

print("\nDuplicate Rows")
print(df.duplicated().sum())

print("\nSummary Statistics")
print(df.describe())

# ============================================================
# STEP 11 : Export
# ============================================================

output = "Gold_DS_FERT_04_Fertilizer_Subsidy.csv"

df.to_csv(output, index=False)

print("\nGold Dataset Created Successfully!")

# ============================================================
# STEP 12 : Download
# ============================================================

files.download(output)

Saving DS-FERT-04_Fertilizer_Subsidy.csv to DS-FERT-04_Fertilizer_Subsidy.csv
ORIGINAL DATASET
              financial_year  fertilizer_subsidy_rs_crore
0                    2024-25                       177129
1  2025-26 (Budget Estimate)                       167887

DATA QUALITY REPORT

Rows : 2

Columns : 4

Missing Values
date                           0
financial_year                 0
year                           0
fertilizer_subsidy_rs_crore    0
dtype: int64

Duplicate Rows
0

Summary Statistics
                      date         year  fertilizer_subsidy_rs_crore
count                    2     2.000000                     2.000000
mean   2024-09-30 12:00:00  2024.500000                172508.000000
min    2024-04-01 00:00:00  2024.000000                167887.000000
25%    2024-07-01 06:00:00  2024.250000                170197.500000
50%    2024-09-30 12:00:00  2024.500000                172508.000000
75%    2024-12-30 18:00:00  2024.750000                174818.500000
max  

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# Hormuz Impact Project
# ETL Module : Fertilizer
# Dataset    : DS-FERT-05_State_Wise_Consumption.csv
# Output     : Gold_DS_FERT_05_State_Wise_Consumption.csv
# Version    : 1.0
# ============================================================

import pandas as pd
from google.colab import files

# ============================================================
# STEP 1 : Upload
# ============================================================

uploaded = files.upload()
filename = list(uploaded.keys())[0]

# ============================================================
# STEP 2 : Read Dataset
# ============================================================

df = pd.read_csv(filename)

print("=" * 70)
print("ORIGINAL DATASET")
print("=" * 70)

print(df.head())

# ============================================================
# STEP 3 : Clean State Names
# ============================================================

df["state"] = (
    df["state"]
      .astype(str)
      .str.strip()
      .str.title()
)

# ============================================================
# STEP 4 : Numeric Conversion
# ============================================================

numeric_cols = [
    "urea_lmt",
    "dap_lmt",
    "mop_lmt",
    "npks_lmt"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# ============================================================
# STEP 5 : Derived KPIs
# ============================================================

df["total_consumption_lakh_mt"] = (
    df["urea_lmt"] +
    df["dap_lmt"] +
    df["mop_lmt"] +
    df["npks_lmt"]
).round(2)

total_consumption = df["total_consumption_lakh_mt"].sum()

df["consumption_share_pct"] = (
    df["total_consumption_lakh_mt"] /
    total_consumption * 100
).round(2)

df["analysis_year"] = 2025

# ============================================================
# STEP 6 : Remove Duplicates
# ============================================================

df.drop_duplicates(inplace=True)

# ============================================================
# STEP 7 : Sort
# ============================================================

df.sort_values(
    "total_consumption_lakh_mt",
    ascending=False,
    inplace=True
)

df.reset_index(drop=True, inplace=True)

# ============================================================
# STEP 8 : Final Column Order
# ============================================================

df = df[
[
    "analysis_year",
    "state",
    "urea_lmt",
    "dap_lmt",
    "mop_lmt",
    "npks_lmt",
    "total_consumption_lakh_mt",
    "consumption_share_pct"
]
]

# ============================================================
# STEP 9 : Validation
# ============================================================

print("\n" + "="*70)
print("DATA QUALITY REPORT")
print("="*70)

print("\nRows :", len(df))
print("\nColumns :", len(df.columns))

print("\nMissing Values")
print(df.isnull().sum())

print("\nDuplicate Rows")
print(df.duplicated().sum())

print("\nSummary Statistics")
print(df.describe())

# ============================================================
# STEP 10 : Export
# ============================================================

output = "Gold_DS_FERT_05_State_Wise_Consumption.csv"

df.to_csv(output, index=False)

print("\nGold Dataset Created Successfully!")

# ============================================================
# STEP 11 : Download
# ============================================================

files.download(output)

Saving DS-FERT-05_State_Wise_Consumption.csv to DS-FERT-05_State_Wise_Consumption.csv
ORIGINAL DATASET
                         state  urea_lmt  dap_lmt  mop_lmt  npks_lmt
0  Andaman and Nicobar Islands      0.00     0.00     0.00      0.00
1               Andhra Pradesh     15.70     4.34     1.71     15.78
2            Arunachal Pradesh      0.01     0.00     0.00      0.00
3                        Assam      4.10     0.62     0.57      0.48
4                        Bihar     25.60     5.83     1.84      6.04

DATA QUALITY REPORT

Rows : 36

Columns : 8

Missing Values
analysis_year                0
state                        0
urea_lmt                     0
dap_lmt                      0
mop_lmt                      0
npks_lmt                     0
total_consumption_lakh_mt    0
consumption_share_pct        0
dtype: int64

Duplicate Rows
0

Summary Statistics
       analysis_year   urea_lmt    dap_lmt    mop_lmt   npks_lmt  \
count           36.0  36.000000  36.000000  36.000000  

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# Hormuz Impact Project
# ETL Module : Forex
# Dataset    : BankWise.xls (HTML Table)
# Output     : Gold_DS_FX_01_USD_INR_Daily.csv
# Version    : 1.0
# ============================================================

import pandas as pd
from google.colab import files

# Upload
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# Read HTML table
df = pd.read_html(filename)[0]

# First row contains headers
df.columns = df.iloc[0]
df = df.iloc[1:].reset_index(drop=True)

# Rename
df.rename(columns={
    "Date": "date",
    "USD (INR / 1 USD)": "usd_inr_rate"
}, inplace=True)

# Date conversion
df["date"] = pd.to_datetime(df["date"], dayfirst=True)

# Numeric conversion
df["usd_inr_rate"] = pd.to_numeric(df["usd_inr_rate"], errors="coerce")

# Calendar columns
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month_name()
df["month_no"] = df["date"].dt.month
df["quarter"] = "Q" + df["date"].dt.quarter.astype(str)
df["week"] = df["date"].dt.isocalendar().week.astype(int)

# Sort by date
df.sort_values("date", inplace=True)

# Derived metrics
df["daily_change"] = df["usd_inr_rate"].diff().round(4)

df["daily_change_pct"] = (
    df["usd_inr_rate"].pct_change() * 100
).round(4)

# Remove duplicates
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

# Final column order
df = df[
    [
        "date",
        "year",
        "month",
        "month_no",
        "quarter",
        "week",
        "usd_inr_rate",
        "daily_change",
        "daily_change_pct"
    ]
]

# Validation
print("=" * 70)
print("DATA QUALITY REPORT")
print("=" * 70)
print(df.info())
print("\nMissing Values")
print(df.isnull().sum())
print("\nDuplicate Rows")
print(df.duplicated().sum())

# Export
output = "Gold_DS_FX_01_USD_INR_Daily.csv"
df.to_csv(output, index=False)

print("\nGold Dataset Created Successfully!")

files.download(output)

Saving BankWise.xls to BankWise.xls
DATA QUALITY REPORT
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 605 entries, 0 to 604
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   date              605 non-null    datetime64[ns]
 1   year              605 non-null    int32         
 2   month             605 non-null    object        
 3   month_no          605 non-null    int32         
 4   quarter           605 non-null    object        
 5   week              605 non-null    int64         
 6   usd_inr_rate      605 non-null    float64       
 7   daily_change      604 non-null    float64       
 8   daily_change_pct  604 non-null    float64       
dtypes: datetime64[ns](1), float64(3), int32(2), int64(1), object(2)
memory usage: 37.9+ KB
None

Missing Values
0
date                0
year                0
month               0
month_no            0
quarter             0
week                0

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
from google.colab import files

# Upload
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# Read sheet
df = pd.read_excel(
    filename,
    sheet_name=0,
    skiprows=6,
    header=None
)

# Assign columns
df.columns = [
    "ignore",
    "year",
    "month",
    "foreign_currency_assets_inr_crore",
    "foreign_currency_assets_usd_million",
    "gold_inr_crore",
    "gold_usd_million",
    "reserve_tranche_position_inr_crore",
    "reserve_tranche_position_usd_million",
    "sdr_million",
    "sdr_inr_crore",
    "sdr_usd_million",
    "total_inr_crore",
    "total_forex_reserves_usd_million"
]

# ---------------------------------------------------
# Keep only actual data rows
# ---------------------------------------------------

# Year must be numeric
df = df[pd.to_numeric(df["year"], errors="coerce").notna()]

# Month must be a valid abbreviation
valid_months = [
    "JAN","FEB","MAR","APR","MAY","JUN",
    "JUL","AUG","SEP","OCT","NOV","DEC"
]

df["month"] = df["month"].astype(str).str.upper().str.strip()

df = df[df["month"].isin(valid_months)]

# ---------------------------------------------------
# Create Date
# ---------------------------------------------------

df["year"] = df["year"].astype(int)

df["date"] = pd.to_datetime(
    df["year"].astype(str) + "-" + df["month"],
    format="%Y-%b"
)

# Calendar columns
df["month_no"] = df["date"].dt.month
df["month"] = df["date"].dt.month_name()
df["quarter"] = "Q" + df["date"].dt.quarter.astype(str)

# ---------------------------------------------------
# Numeric conversion
# ---------------------------------------------------

numeric_cols = [
    "foreign_currency_assets_usd_million",
    "gold_usd_million",
    "reserve_tranche_position_usd_million",
    "sdr_usd_million",
    "total_forex_reserves_usd_million"
]

for col in numeric_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(",", "", regex=False)
    )
    df[col] = pd.to_numeric(df[col], errors="coerce")

# ---------------------------------------------------
# Derived KPIs
# ---------------------------------------------------

df["gold_share_pct"] = (
    df["gold_usd_million"] /
    df["total_forex_reserves_usd_million"] * 100
).round(2)

df["fca_share_pct"] = (
    df["foreign_currency_assets_usd_million"] /
    df["total_forex_reserves_usd_million"] * 100
).round(2)

# ---------------------------------------------------
# Final columns
# ---------------------------------------------------

df = df[
    [
        "date",
        "year",
        "month",
        "month_no",
        "quarter",
        "foreign_currency_assets_usd_million",
        "gold_usd_million",
        "reserve_tranche_position_usd_million",
        "sdr_usd_million",
        "total_forex_reserves_usd_million",
        "gold_share_pct",
        "fca_share_pct"
    ]
]

df = df.sort_values("date").reset_index(drop=True)

print(df.head())
print(df.tail())
print(df.info())

output = "Gold_DS_FX_02_Forex_Reserves.csv"
df.to_csv(output, index=False)

files.download(output)

Saving Foreign Exchange Reserves.xlsx to Foreign Exchange Reserves (3).xlsx
        date  year  month  month_no quarter  \
0 1990-03-01  1990  March         3      Q1   
1 1990-04-01  1990  April         4      Q2   
2 1990-05-01  1990    May         5      Q2   
3 1990-06-01  1990   June         6      Q2   
4 1990-07-01  1990   July         7      Q3   

   foreign_currency_assets_usd_million  gold_usd_million  \
0                               3368.0             487.0   
1                               2949.0             487.0   
2                               3096.0             491.0   
3                               3072.0             495.0   
4                               2920.0             511.0   

   reserve_tranche_position_usd_million  sdr_usd_million  \
0                                   NaN            107.0   
1                                   NaN            189.0   
2                                   NaN            114.0   
3                                   NaN 

/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# Hormuz Impact Project
# ETL Module : Forex
# Dataset    : DS-FX-04_NIFTY50.csv
# Output     : Gold_DS_FX_04_NIFTY50.csv
# Version    : 1.0
# ============================================================

import pandas as pd
from google.colab import files

# Upload
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# Read CSV
df = pd.read_csv(filename)

# ------------------------------------------------------------
# Clean column names
# ------------------------------------------------------------

df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
      .str.replace("(₹_cr)", "rs_crore", regex=False)
)

# Rename turnover column
df.rename(columns={
    "turnover_(₹_cr)": "turnover_rs_crore"
}, inplace=True)

# ------------------------------------------------------------
# Date
# ------------------------------------------------------------

df["date"] = pd.to_datetime(df["date"], format="%d-%b-%y")

df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month_name()
df["month_no"] = df["date"].dt.month
df["quarter"] = "Q" + df["date"].dt.quarter.astype(str)

# ------------------------------------------------------------
# Numeric
# ------------------------------------------------------------

numeric_cols = [
    "open",
    "high",
    "low",
    "close",
    "shares_traded",
    "turnover_rs_crore"
]

for col in numeric_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(",", "", regex=False)
    )
    df[col] = pd.to_numeric(df[col], errors="coerce")

# ------------------------------------------------------------
# Sort
# ------------------------------------------------------------

df = df.sort_values("date").reset_index(drop=True)

# ------------------------------------------------------------
# Business KPIs
# ------------------------------------------------------------

df["daily_change"] = df["close"].diff().round(2)

df["daily_return_pct"] = (
    df["close"].pct_change() * 100
).round(2)

df["intraday_range"] = (
    df["high"] - df["low"]
).round(2)

# ------------------------------------------------------------
# Remove duplicates
# ------------------------------------------------------------

df.drop_duplicates(inplace=True)

# ------------------------------------------------------------
# Final Columns
# ------------------------------------------------------------

df = df[
    [
        "date",
        "year",
        "month",
        "month_no",
        "quarter",
        "open",
        "high",
        "low",
        "close",
        "shares_traded",
        "turnover_rs_crore",
        "daily_change",
        "daily_return_pct",
        "intraday_range"
    ]
]

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("="*70)
print("DATA QUALITY REPORT")
print("="*70)

print(df.info())
print("\nMissing Values")
print(df.isnull().sum())

print("\nDuplicate Rows")
print(df.duplicated().sum())

# ------------------------------------------------------------
# Export
# ------------------------------------------------------------

output = "Gold_DS_FX_04_NIFTY50.csv"

df.to_csv(output, index=False)

files.download(output)

Saving NIFTY 50-01-01-2025-to-31-12-2025.csv to NIFTY 50-01-01-2025-to-31-12-2025.csv
DATA QUALITY REPORT
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 497 entries, 0 to 496
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   date               497 non-null    datetime64[ns]
 1   year               497 non-null    int32         
 2   month              497 non-null    object        
 3   month_no           497 non-null    int32         
 4   quarter            497 non-null    object        
 5   open               497 non-null    float64       
 6   high               497 non-null    float64       
 7   low                497 non-null    float64       
 8   close              497 non-null    float64       
 9   shares_traded      497 non-null    int64         
 10  turnover_rs_crore  497 non-null    float64       
 11  daily_change       496 non-null    float64       
 12  daily_return_p

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# Hormuz Impact Project
# ETL Module : Forex
# Dataset    : DS-FX-05_Sensex_Daily.csv
# Output     : Gold_DS_FX_05_Sensex_Daily.csv
# Version    : 1.0
# ============================================================

import pandas as pd
from google.colab import files

# ============================================================
# STEP 1 : Upload
# ============================================================

uploaded = files.upload()
filename = list(uploaded.keys())[0]

# ============================================================
# STEP 2 : Read Dataset
# ============================================================

df = pd.read_csv(filename)

print("=" * 70)
print("ORIGINAL DATASET")
print("=" * 70)

print(df.head())

# ============================================================
# STEP 3 : Standardize Column Names
# ============================================================

df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
)

# ============================================================
# STEP 4 : Date
# ============================================================

# Remove timezone information if present
df["date"] = pd.to_datetime(df["date"]).dt.tz_localize(None)

df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month_name()
df["month_no"] = df["date"].dt.month
df["quarter"] = "Q" + df["date"].dt.quarter.astype(str)

# ============================================================
# STEP 5 : Numeric Conversion
# ============================================================

numeric_cols = [
    "open",
    "high",
    "low",
    "close",
    "volume"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# ============================================================
# STEP 6 : Derived KPIs
# ============================================================

df["daily_change"] = df["close"].diff().round(2)

df["daily_return_pct"] = (
    df["close"].pct_change() * 100
).round(2)

df["intraday_range"] = (
    df["high"] - df["low"]
).round(2)

# ============================================================
# STEP 7 : Remove Unnecessary Columns
# ============================================================

drop_cols = ["adj_close", "dividends", "stock_splits"]

df.drop(columns=drop_cols, inplace=True, errors="ignore")

# ============================================================
# STEP 8 : Remove Duplicates & Sort
# ============================================================

df.drop_duplicates(inplace=True)
df.sort_values("date", inplace=True)
df.reset_index(drop=True, inplace=True)

# ============================================================
# STEP 9 : Final Column Order
# ============================================================

df = df[
    [
        "date",
        "year",
        "month",
        "month_no",
        "quarter",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "daily_change",
        "daily_return_pct",
        "intraday_range"
    ]
]

# ============================================================
# STEP 10 : Validation
# ============================================================

print("\n" + "=" * 70)
print("DATA QUALITY REPORT")
print("=" * 70)

print("\nRows :", len(df))
print("\nColumns :", len(df.columns))

print("\nMissing Values")
print(df.isnull().sum())

print("\nDuplicate Rows")
print(df.duplicated().sum())

print("\nSummary Statistics")
print(df.describe())

# ============================================================
# STEP 11 : Export
# ============================================================

output = "Gold_DS_FX_05_Sensex_Daily.csv"

df.to_csv(output, index=False)

print("\nGold Dataset Created Successfully!")

# ============================================================
# STEP 12 : Download
# ============================================================

files.download(output)

Saving DS-FX-04_Sensex_Daily.csv to DS-FX-04_Sensex_Daily.csv
ORIGINAL DATASET
                        Date          Open          High           Low  \
0  2024-04-01 00:00:00+05:30  73968.617188  74254.617188  73909.390625   
1  2024-04-02 00:00:00+05:30  74022.296875  74099.781250  73743.773438   
2  2024-04-03 00:00:00+05:30  73757.226562  74151.210938  73540.273438   
3  2024-04-04 00:00:00+05:30  74413.820312  74501.726562  73485.117188   
4  2024-04-05 00:00:00+05:30  74287.023438  74361.109375  73946.921875   

          Close     Adj Close  Volume  Dividends  Stock Splits  
0  74014.546875  74014.546875       0        0.0           0.0  
1  73903.906250  73903.906250    6300        0.0           0.0  
2  73876.820312  73876.820312    7900        0.0           0.0  
3  74227.632812  74227.632812   10700        0.0           0.0  
4  74248.218750  74248.218750   11300        0.0           0.0  

DATA QUALITY REPORT

Rows : 493

Columns : 13

Missing Values
date                0
y

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# Hormuz Impact Project
# ETL Module : Forex
# Dataset    : DS-FX-05_India_VIX_Daily.csv
# Output     : Gold_DS_FX_06_India_VIX.csv
# Version    : 1.0
# ============================================================

import pandas as pd
from google.colab import files

# ============================================================
# STEP 1 : Upload
# ============================================================

uploaded = files.upload()
filename = list(uploaded.keys())[0]

# ============================================================
# STEP 2 : Read Dataset
# ============================================================

df = pd.read_csv(filename)

print("=" * 70)
print("ORIGINAL DATASET")
print("=" * 70)

print(df.head())

# ============================================================
# STEP 3 : Standardize Column Names
# ============================================================

df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
)

# ============================================================
# STEP 4 : Date
# ============================================================

df["date"] = pd.to_datetime(df["date"])

df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month_name()
df["month_no"] = df["date"].dt.month
df["quarter"] = "Q" + df["date"].dt.quarter.astype(str)

# ============================================================
# STEP 5 : Numeric Conversion
# ============================================================

numeric_cols = [
    "open",
    "high",
    "low",
    "close",
    "volume"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# ============================================================
# STEP 6 : Derived KPIs
# ============================================================

df["daily_change"] = df["close"].diff().round(2)

df["daily_return_pct"] = (
    df["close"].pct_change() * 100
).round(2)

df["intraday_range"] = (
    df["high"] - df["low"]
).round(2)

# ============================================================
# STEP 7 : Remove Duplicates & Sort
# ============================================================

df.drop_duplicates(inplace=True)

df.sort_values("date", inplace=True)

df.reset_index(drop=True, inplace=True)

# ============================================================
# STEP 8 : Final Column Order
# ============================================================

df = df[
    [
        "date",
        "year",
        "month",
        "month_no",
        "quarter",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "daily_change",
        "daily_return_pct",
        "intraday_range"
    ]
]

# ============================================================
# STEP 9 : Validation
# ============================================================

print("\n" + "=" * 70)
print("DATA QUALITY REPORT")
print("=" * 70)

print(df.info())

print("\nMissing Values")
print(df.isnull().sum())

print("\nDuplicate Rows")
print(df.duplicated().sum())

print("\nSummary Statistics")
print(df.describe())

# ============================================================
# STEP 10 : Export
# ============================================================

output = "Gold_DS_FX_06_India_VIX.csv"

df.to_csv(output, index=False)

print("\nGold Dataset Created Successfully!")

# ============================================================
# STEP 11 : Download
# ============================================================

files.download(output)

Saving DS-FX-05_India_VIX_Daily.csv to DS-FX-05_India_VIX_Daily.csv
ORIGINAL DATASET
         date   open   high    low  close  volume
0  2024-04-01  12.83  13.36  12.01  12.08       0
1  2024-04-02  12.08  12.31  11.60  11.65       0
2  2024-04-03  11.65  11.83  11.31  11.37       0
3  2024-04-04  11.37  11.84  10.86  11.22       0
4  2024-04-05  11.22  11.68  11.04  11.34       0

DATA QUALITY REPORT
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 491 entries, 0 to 490
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   date              491 non-null    datetime64[ns]
 1   year              491 non-null    int32         
 2   month             491 non-null    object        
 3   month_no          491 non-null    int32         
 4   quarter           491 non-null    object        
 5   open              491 non-null    float64       
 6   high              491 non-null    float64       
 7 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# Hormuz Impact Project
# ETL Module : Forex
# Dataset    : DS-INF-06_RBI_Policy_Repo_Rate.csv
# Output     : Gold_DS_FX_03_RBI_Repo_Rate.csv
# Version    : 1.0
# ============================================================

import pandas as pd
from google.colab import files

# ============================================================
# STEP 1 : Upload
# ============================================================

uploaded = files.upload()
filename = list(uploaded.keys())[0]

# ============================================================
# STEP 2 : Read Dataset
# ============================================================

df = pd.read_csv(filename)

print("=" * 70)
print("ORIGINAL DATASET")
print("=" * 70)

print(df.head())

# ============================================================
# STEP 3 : Date
# ============================================================

df["date"] = pd.to_datetime(df["date"])

df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month_name()
df["month_no"] = df["date"].dt.month
df["quarter"] = "Q" + df["date"].dt.quarter.astype(str)

# ============================================================
# STEP 4 : Numeric
# ============================================================

df["repo_rate"] = pd.to_numeric(df["repo_rate"], errors="coerce")

# ============================================================
# STEP 5 : Sort
# ============================================================

df = df.sort_values("date").reset_index(drop=True)

# ============================================================
# STEP 6 : Derived KPI
# ============================================================

df["repo_rate_change"] = df["repo_rate"].diff().round(2)

# ============================================================
# STEP 7 : Remove Duplicates
# ============================================================

df.drop_duplicates(inplace=True)

# ============================================================
# STEP 8 : Final Columns
# ============================================================

df = df[
    [
        "date",
        "year",
        "month",
        "month_no",
        "quarter",
        "repo_rate",
        "repo_rate_change"
    ]
]

# ============================================================
# STEP 9 : Validation
# ============================================================

print("\n" + "=" * 70)
print("DATA QUALITY REPORT")
print("=" * 70)

print(df.info())

print("\nMissing Values")
print(df.isnull().sum())

print("\nDuplicate Rows")
print(df.duplicated().sum())

print("\nSummary Statistics")
print(df.describe())

# ============================================================
# STEP 10 : Export
# ============================================================

output = "Gold_DS_FX_03_RBI_Repo_Rate.csv"

df.to_csv(output, index=False)

print("\nGold Dataset Created Successfully!")

# ============================================================
# STEP 11 : Download
# ============================================================

files.download(output)

Saving DS-INF-06_RBI_Policy_Repo_Rate.csv to DS-INF-06_RBI_Policy_Repo_Rate.csv
ORIGINAL DATASET
         date  year     month  repo_rate
0  2024-01-01  2024   January        6.5
1  2024-02-01  2024  February        6.5
2  2024-03-01  2024     March        6.5
3  2024-04-01  2024     April        6.5
4  2024-05-01  2024       May        6.5

DATA QUALITY REPORT
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   date              24 non-null     datetime64[ns]
 1   year              24 non-null     int32         
 2   month             24 non-null     object        
 3   month_no          24 non-null     int32         
 4   quarter           24 non-null     object        
 5   repo_rate         24 non-null     float64       
 6   repo_rate_change  23 non-null     float64       
dtypes: datetime64[ns](1), float64(2), int32(2),

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# Hormuz Impact Project
# ADS Module : Forex
# Dataset    : ADS_FX_01_Forex_Market_Analysis.csv
# Version    : 1.0
# ============================================================

import pandas as pd
import numpy as np
from google.colab import files

# ============================================================
# STEP 1 : Upload Gold Datasets
# ============================================================

print("Upload the following files:")
print("1. Gold_DS_FX_01_USD_INR_Daily.csv")
print("2. Gold_DS_FX_02_Forex_Reserves.csv")
print("3. Gold_DS_FX_03_RBI_Repo_Rate.csv")
print("4. Gold_DS_FX_04_NIFTY50.csv")
print("5. Gold_DS_FX_05_Sensex_Daily.csv")
print("6. Gold_DS_FX_06_India_VIX.csv")

uploaded = files.upload()

# ============================================================
# STEP 2 : Read Files
# ============================================================

usd = None
reserves = None
repo = None
nifty = None
sensex = None
vix = None

for file in uploaded.keys():

    if "FX_01" in file:
        usd = pd.read_csv(file)

    elif "FX_02" in file:
        reserves = pd.read_csv(file)

    elif "FX_03" in file:
        repo = pd.read_csv(file)

    elif "FX_04" in file:
        nifty = pd.read_csv(file)

    elif "FX_05" in file:
        sensex = pd.read_csv(file)

    elif "FX_06" in file:
        vix = pd.read_csv(file)

# ============================================================
# STEP 3 : Date Conversion
# ============================================================

datasets = [usd, reserves, repo, nifty, sensex, vix]

for df in datasets:
    df["date"] = pd.to_datetime(df["date"])

# ============================================================
# STEP 4 : Convert Daily Data → Monthly
# ============================================================

# USD/INR -> Month End
usd_monthly = (
    usd.sort_values("date")
       .groupby(pd.Grouper(key="date", freq="MS"))
       .last()
       .reset_index()
)

usd_monthly = usd_monthly[
    ["date","usd_inr_rate"]
]

# NIFTY -> Month End Close
nifty_monthly = (
    nifty.sort_values("date")
         .groupby(pd.Grouper(key="date", freq="MS"))
         .last()
         .reset_index()
)

nifty_monthly = nifty_monthly[
    ["date","close"]
].rename(columns={
    "close":"nifty_close"
})

# Sensex -> Month End Close
sensex_monthly = (
    sensex.sort_values("date")
           .groupby(pd.Grouper(key="date", freq="MS"))
           .last()
           .reset_index()
)

sensex_monthly = sensex_monthly[
    ["date","close"]
].rename(columns={
    "close":"sensex_close"
})

# India VIX -> Monthly Average
vix_monthly = (
    vix.groupby(pd.Grouper(key="date",freq="MS"))
       .agg({
           "close":"mean"
       })
       .reset_index()
)

vix_monthly.rename(columns={
    "close":"india_vix_avg"
}, inplace=True)

# Repo Rate -> Monthly Forward Fill
repo_monthly = repo.copy()

repo_monthly = (
    repo_monthly
      .set_index("date")
      .resample("MS")
      .ffill()
      .reset_index()
)

repo_monthly = repo_monthly[
    ["date","repo_rate"]
]

# Forex Reserves
reserves_monthly = reserves[
[
"date",
"total_forex_reserves_usd_million"
]
]

# ============================================================
# STEP 5 : Merge
# ============================================================

df = usd_monthly.merge(
    reserves_monthly,
    on="date",
    how="outer"
)

df = df.merge(
    repo_monthly,
    on="date",
    how="outer"
)

df = df.merge(
    nifty_monthly,
    on="date",
    how="outer"
)

df = df.merge(
    sensex_monthly,
    on="date",
    how="outer"
)

df = df.merge(
    vix_monthly,
    on="date",
    how="outer"
)

# ============================================================
# STEP 6 : Calendar Columns
# ============================================================

df.sort_values("date", inplace=True)

df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month_name()
df["month_no"] = df["date"].dt.month
df["quarter"] = "Q" + df["date"].dt.quarter.astype(str)

# ============================================================
# STEP 7 : Forward Fill Monthly Series
# ============================================================

df["repo_rate"] = df["repo_rate"].ffill()

df["total_forex_reserves_usd_million"] = (
    df["total_forex_reserves_usd_million"]
      .ffill()
)

# ============================================================
# STEP 8 : Business KPIs
# ============================================================

df["usd_inr_monthly_change_pct"] = (
    df["usd_inr_rate"].pct_change()*100
).round(2)

df["forex_reserve_growth_pct"] = (
    df["total_forex_reserves_usd_million"]
      .pct_change()*100
).round(2)

df["nifty_monthly_return_pct"] = (
    df["nifty_close"].pct_change()*100
).round(2)

df["sensex_monthly_return_pct"] = (
    df["sensex_close"].pct_change()*100
).round(2)

# ============================================================
# STEP 9 : Market Sentiment
# ============================================================

conditions = [
    df["india_vix_avg"] < 15,
    (df["india_vix_avg"] >= 15) &
    (df["india_vix_avg"] < 20),
    df["india_vix_avg"] >= 20
]

choices = [
    "Bullish",
    "Neutral",
    "Risk-Off"
]

df["market_sentiment"] = np.select(
    conditions,
    choices,
    default="Neutral"
)

# ============================================================
# STEP 10 : Final Columns
# ============================================================

df = df[
[
"date",
"year",
"month",
"month_no",
"quarter",

"usd_inr_rate",
"total_forex_reserves_usd_million",
"repo_rate",

"nifty_close",
"sensex_close",
"india_vix_avg",

"usd_inr_monthly_change_pct",
"forex_reserve_growth_pct",
"nifty_monthly_return_pct",
"sensex_monthly_return_pct",

"market_sentiment"
]
]

# ============================================================
# STEP 11 : Validation
# ============================================================

print("="*70)
print("DATA QUALITY REPORT")
print("="*70)

print(df.info())

print("\nMissing Values")
print(df.isnull().sum())

print("\nDuplicate Rows")
print(df.duplicated().sum())

print("\nSummary Statistics")
print(df.describe())

# ============================================================
# STEP 12 : Export
# ============================================================

output = "ADS_FX_01_Forex_Market_Analysis.csv"

df.to_csv(output,index=False)

print("\nADS Dataset Created Successfully!")

files.download(output)

Upload the following files:
1. Gold_DS_FX_01_USD_INR_Daily.csv
2. Gold_DS_FX_02_Forex_Reserves.csv
3. Gold_DS_FX_03_RBI_Repo_Rate.csv
4. Gold_DS_FX_04_NIFTY50.csv
5. Gold_DS_FX_05_Sensex_Daily.csv
6. Gold_DS_FX_06_India_VIX.csv


Saving Gold_DS_FX_03_RBI_Repo_Rate.csv to Gold_DS_FX_03_RBI_Repo_Rate (1).csv
Saving Gold_DS_FX_06_India_VIX.csv to Gold_DS_FX_06_India_VIX (1).csv
Saving Gold_DS_FX_05_Sensex_Daily.csv to Gold_DS_FX_05_Sensex_Daily (1).csv
Saving Gold_DS_FX_04_NIFTY50.csv to Gold_DS_FX_04_NIFTY50 (1).csv
Saving Gold_DS_FX_02_Forex_Reserves.csv to Gold_DS_FX_02_Forex_Reserves (1).csv
Saving Gold_DS_FX_01_USD_INR_Daily.csv to Gold_DS_FX_01_USD_INR_Daily (1).csv
DATA QUALITY REPORT
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 437 entries, 0 to 436
Data columns (total 16 columns):
 #   Column                            Non-Null Count  Dtype         
---  ------                            --------------  -----         
 0   date                              437 non-null    datetime64[ns]
 1   year                              437 non-null    int32         
 2   month                             437 non-null    object        
 3   month_no                          437 non-null    int32         
 4   qu

/tmp/ipykernel_2337/2913840199.py:216: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df["nifty_close"].pct_change()*100
/tmp/ipykernel_2337/2913840199.py:220: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df["sensex_close"].pct_change()*100


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# Hormuz Impact Project
# ETL Module : Shipping
# Dataset    : DS-SHIP-02_Freight_Rates.csv
# Output     : Gold_DS_SHIP_01_Freight_Rates.csv
# Version    : 1.0
# ============================================================

import pandas as pd
from google.colab import files

# ============================================================
# STEP 1 : Upload
# ============================================================

uploaded = files.upload()
filename = list(uploaded.keys())[0]

# ============================================================
# STEP 2 : Read Dataset
# ============================================================

df = pd.read_csv(filename)

print("=" * 70)
print("ORIGINAL DATASET")
print("=" * 70)

print(df.head())

# ============================================================
# STEP 3 : Date
# ============================================================

df["date"] = pd.to_datetime(df["month"] + "-01")

df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month_name()
df["month_no"] = df["date"].dt.month
df["quarter"] = "Q" + df["date"].dt.quarter.astype(str)

# ============================================================
# STEP 4 : Standardize Route
# ============================================================

df["route"] = (
    df["route"]
      .astype(str)
      .str.strip()
      .str.title()
)

# ============================================================
# STEP 5 : Numeric Conversion
# ============================================================

df["freight_rate_usd_per_day"] = pd.to_numeric(
    df["freight_rate_usd_per_day"],
    errors="coerce"
)

# ============================================================
# STEP 6 : Sort
# ============================================================

df = df.sort_values("date").reset_index(drop=True)

# ============================================================
# STEP 7 : Derived KPIs
# ============================================================

df["monthly_change_usd"] = (
    df["freight_rate_usd_per_day"].diff()
).round(2)

df["monthly_change_pct"] = (
    df["freight_rate_usd_per_day"].pct_change() * 100
).round(2)

# ============================================================
# STEP 8 : Remove Duplicates
# ============================================================

df.drop_duplicates(inplace=True)

# ============================================================
# STEP 9 : Final Columns
# ============================================================

df = df[
    [
        "date",
        "year",
        "month",
        "month_no",
        "quarter",
        "route",
        "freight_rate_usd_per_day",
        "monthly_change_usd",
        "monthly_change_pct"
    ]
]

# ============================================================
# STEP 10 : Validation
# ============================================================

print("\n" + "=" * 70)
print("DATA QUALITY REPORT")
print("=" * 70)

print(df.info())

print("\nMissing Values")
print(df.isnull().sum())

print("\nDuplicate Rows")
print(df.duplicated().sum())

print("\nSummary Statistics")
print(df.describe())

# ============================================================
# STEP 11 : Export
# ============================================================

output = "Gold_DS_SHIP_01_Freight_Rates.csv"

df.to_csv(output, index=False)

print("\nGold Dataset Created Successfully!")

# ============================================================
# STEP 12 : Download
# ============================================================

files.download(output)

Saving DS-SHIP-02_Freight_Rates.csv to DS-SHIP-02_Freight_Rates.csv
ORIGINAL DATASET
     month              route  freight_rate_usd_per_day
0  2024-04  Middle East-India                     27800
1  2024-05  Middle East-India                     28150
2  2024-06  Middle East-India                     28500
3  2024-07  Middle East-India                     28900
4  2024-08  Middle East-India                     29250

DATA QUALITY REPORT
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   date                      15 non-null     datetime64[ns]
 1   year                      15 non-null     int32         
 2   month                     15 non-null     object        
 3   month_no                  15 non-null     int32         
 4   quarter                   15 non-null     object        
 5   route            

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# Hormuz Impact Project
# ETL Module : Shipping
# Dataset    : DS-SHIP-03_Marine_Insurance_Premiums_Synthetic.csv
# Output     : Gold_DS_SHIP_02_Marine_Insurance.csv
# Version    : 1.0
# ============================================================

import pandas as pd
from google.colab import files

# ============================================================
# STEP 1 : Upload
# ============================================================

uploaded = files.upload()
filename = list(uploaded.keys())[0]

# ============================================================
# STEP 2 : Read Dataset
# ============================================================

df = pd.read_csv(filename)

print("="*70)
print("ORIGINAL DATASET")
print("="*70)
print(df.head())

# ============================================================
# STEP 3 : Create Date
# ============================================================

df["date"] = pd.to_datetime(df["month"] + "-01")

df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month_name()
df["month_no"] = df["date"].dt.month
df["quarter"] = "Q" + df["date"].dt.quarter.astype(str)

# ============================================================
# STEP 4 : Clean Text Columns
# ============================================================

df["financial_year"] = (
    df["financial_year"]
    .astype(str)
    .str.strip()
)

df["region"] = (
    df["region"]
    .astype(str)
    .str.strip()
    .str.title()
)

df["remarks"] = (
    df["remarks"]
    .astype(str)
    .str.strip()
)

# ============================================================
# STEP 5 : Numeric Conversion
# ============================================================

df["war_risk_premium_percent"] = pd.to_numeric(
    df["war_risk_premium_percent"],
    errors="coerce"
)

# ============================================================
# STEP 6 : Sort
# ============================================================

df = df.sort_values("date").reset_index(drop=True)

# ============================================================
# STEP 7 : Derived KPIs
# ============================================================

df["premium_change_pct_points"] = (
    df["war_risk_premium_percent"].diff()
).round(2)

df["premium_growth_pct"] = (
    df["war_risk_premium_percent"].pct_change() * 100
).round(2)

# ============================================================
# STEP 8 : Remove Duplicates
# ============================================================

df.drop_duplicates(inplace=True)

# ============================================================
# STEP 9 : Final Columns
# ============================================================

df = df[
    [
        "date",
        "financial_year",
        "year",
        "month",
        "month_no",
        "quarter",
        "region",
        "war_risk_premium_percent",
        "premium_change_pct_points",
        "premium_growth_pct",
        "remarks"
    ]
]

# ============================================================
# STEP 10 : Validation
# ============================================================

print("\n" + "="*70)
print("DATA QUALITY REPORT")
print("="*70)

print(df.info())

print("\nMissing Values")
print(df.isnull().sum())

print("\nDuplicate Rows")
print(df.duplicated().sum())

print("\nSummary Statistics")
print(df.describe())

# ============================================================
# STEP 11 : Export
# ============================================================

output = "Gold_DS_SHIP_02_Marine_Insurance.csv"

df.to_csv(output, index=False)

print("\nGold Dataset Created Successfully!")

# ============================================================
# STEP 12 : Download
# ============================================================

files.download(output)

Saving DS-SHIP-03_Marine_Insurance_Premiums_Synthetic.csv to DS-SHIP-03_Marine_Insurance_Premiums_Synthetic.csv
ORIGINAL DATASET
     month financial_year            region  war_risk_premium_percent  \
0  2024-04        2024-25  Middle East Gulf                      0.22   
1  2024-05        2024-25  Middle East Gulf                      0.22   
2  2024-06        2024-25  Middle East Gulf                      0.22   
3  2024-07        2024-25  Middle East Gulf                      0.23   
4  2024-08        2024-25  Middle East Gulf                      0.23   

                    remarks  
0  Stable market conditions  
1  Stable market conditions  
2  Stable market conditions  
3         Regional tensions  
4             Stable market  

DATA QUALITY REPORT
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# Hormuz Impact Project
# ETL Module : Shipping
# Dataset    : DS-SHIP-04_Transit_Time_Route_Delay.csv
# Output     : Gold_DS_SHIP_03_Transit_Time_Delay.csv
# ============================================================

import pandas as pd
from google.colab import files

# Upload
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# Read
df = pd.read_csv(filename)

print("=" * 70)
print("ORIGINAL DATASET")
print("=" * 70)
print(df.head())

# ------------------------------------------------------------
# Date
# ------------------------------------------------------------

df["date"] = pd.to_datetime(df["month"] + "-01")

df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month_name()
df["month_no"] = df["date"].dt.month
df["quarter"] = "Q" + df["date"].dt.quarter.astype(str)

# ------------------------------------------------------------
# Clean Text
# ------------------------------------------------------------

df["financial_year"] = df["financial_year"].astype(str).str.strip()

df["route"] = (
    df["route"]
      .astype(str)
      .str.strip()
      .str.title()
)

df["remarks"] = df["remarks"].astype(str).str.strip()

# ------------------------------------------------------------
# Numeric
# ------------------------------------------------------------

numeric_cols = [
    "average_transit_days",
    "average_delay_days"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# ------------------------------------------------------------
# Derived KPIs
# ------------------------------------------------------------

df["total_transit_days"] = (
    df["average_transit_days"] +
    df["average_delay_days"]
)

df["delay_percentage"] = (
    df["average_delay_days"] /
    df["average_transit_days"] * 100
).round(2)

# ------------------------------------------------------------
# Remove Duplicates & Sort
# ------------------------------------------------------------

df.drop_duplicates(inplace=True)

df.sort_values("date", inplace=True)

df.reset_index(drop=True, inplace=True)

# ------------------------------------------------------------
# Final Columns
# ------------------------------------------------------------

df = df[
[
    "date",
    "financial_year",
    "year",
    "month",
    "month_no",
    "quarter",
    "route",
    "average_transit_days",
    "average_delay_days",
    "total_transit_days",
    "delay_percentage",
    "remarks"
]
]

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DATA QUALITY REPORT")
print("=" * 70)

print(df.info())

print("\nMissing Values")
print(df.isnull().sum())

print("\nDuplicate Rows")
print(df.duplicated().sum())

print("\nSummary Statistics")
print(df.describe())

# ------------------------------------------------------------
# Export
# ------------------------------------------------------------

output = "Gold_DS_SHIP_03_Transit_Time_Delay.csv"

df.to_csv(output, index=False)

print("\nGold Dataset Created Successfully!")

files.download(output)

Saving DS-SHIP-04_Transit_Time_Route_Delay.csv to DS-SHIP-04_Transit_Time_Route_Delay.csv
ORIGINAL DATASET
     month financial_year              route  average_transit_days  \
0  2024-04        2024-25  Middle East-India                    12   
1  2024-05        2024-25  Middle East-India                    12   
2  2024-06        2024-25  Middle East-India                    12   
3  2024-07        2024-25  Middle East-India                    12   
4  2024-08        2024-25  Middle East-India                    13   

   average_delay_days            remarks  
0                   0  Normal operations  
1                   0  Normal operations  
2                   0  Normal operations  
3                   1   Minor congestion  
4                   1   Seasonal traffic  

DATA QUALITY REPORT
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                -----

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# Hormuz Impact Project
# ADS Module : Shipping
# Dataset    : ADS_SHIP_01_Maritime_Risk_Analysis.csv
# Version    : 1.0
# ============================================================

import pandas as pd
import numpy as np
from google.colab import files

# ============================================================
# STEP 1 : Upload Gold Datasets
# ============================================================

print("Upload:")
print("1. Gold_DS_SHIP_01_Freight_Rates.csv")
print("2. Gold_DS_SHIP_02_Marine_Insurance.csv")
print("3. Gold_DS_SHIP_03_Transit_Time_Delay.csv")

uploaded = files.upload()

# ============================================================
# STEP 2 : Read Files
# ============================================================

freight = None
insurance = None
transit = None

for file in uploaded.keys():

    if "SHIP_01" in file:
        freight = pd.read_csv(file)

    elif "SHIP_02" in file:
        insurance = pd.read_csv(file)

    elif "SHIP_03" in file:
        transit = pd.read_csv(file)

# ============================================================
# STEP 3 : Date Conversion
# ============================================================

for df in [freight, insurance, transit]:
    df["date"] = pd.to_datetime(df["date"])

# ============================================================
# STEP 4 : Merge
# ============================================================

df = freight.merge(
    insurance[
        [
            "date",
            "war_risk_premium_percent"
        ]
    ],
    on="date",
    how="outer"
)

df = df.merge(
    transit[
        [
            "date",
            "average_transit_days",
            "average_delay_days",
            "total_transit_days",
            "delay_percentage"
        ]
    ],
    on="date",
    how="outer"
)

# ============================================================
# STEP 5 : Calendar Columns
# ============================================================

df = df.sort_values("date")

df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month_name()
df["month_no"] = df["date"].dt.month
df["quarter"] = "Q" + df["date"].dt.quarter.astype(str)

# ============================================================
# STEP 6 : Forward Fill
# ============================================================

df["war_risk_premium_percent"] = (
    df["war_risk_premium_percent"].ffill()
)

# ============================================================
# STEP 7 : Business KPIs
# ============================================================

df["freight_growth_pct"] = (
    df["freight_rate_usd_per_day"].pct_change()*100
).round(2)

df["insurance_growth_pct"] = (
    df["war_risk_premium_percent"].pct_change()*100
).round(2)

# ============================================================
# STEP 8 : Maritime Risk Score
# ============================================================

# Normalize each component (0–100)

freight_norm = (
    df["freight_growth_pct"]
      .fillna(0)
      .clip(lower=0)
)

insurance_norm = (
    df["war_risk_premium_percent"]
      .fillna(0)
)

delay_norm = (
    df["delay_percentage"]
      .fillna(0)
)

df["maritime_risk_score"] = (
      freight_norm * 0.30
    + insurance_norm * 0.40
    + delay_norm * 0.30
).round(2)

# ============================================================
# STEP 9 : Maritime Risk Level
# ============================================================

conditions = [

    df["maritime_risk_score"] < 15,

    (df["maritime_risk_score"] >= 15) &
    (df["maritime_risk_score"] < 35),

    df["maritime_risk_score"] >= 35

]

choices = [

    "Low",

    "Medium",

    "High"

]

df["maritime_risk_level"] = np.select(
    conditions,
    choices,
    default="Medium"
)

# ============================================================
# STEP 10 : Final Columns
# ============================================================

df = df[
[
"date",
"year",
"month",
"month_no",
"quarter",

"route",

"freight_rate_usd_per_day",

"war_risk_premium_percent",

"average_transit_days",
"average_delay_days",
"total_transit_days",
"delay_percentage",

"freight_growth_pct",
"insurance_growth_pct",

"maritime_risk_score",
"maritime_risk_level"

]
]

# ============================================================
# STEP 11 : Validation
# ============================================================

print("="*70)
print("DATA QUALITY REPORT")
print("="*70)

print(df.info())

print("\nMissing Values")
print(df.isnull().sum())

print("\nDuplicate Rows")
print(df.duplicated().sum())

print("\nSummary Statistics")
print(df.describe())

# ============================================================
# STEP 12 : Export
# ============================================================

output = "ADS_SHIP_01_Maritime_Risk_Analysis.csv"

df.to_csv(output,index=False)

print("\nADS Dataset Created Successfully!")

files.download(output)

Upload:
1. Gold_DS_SHIP_01_Freight_Rates.csv
2. Gold_DS_SHIP_02_Marine_Insurance.csv
3. Gold_DS_SHIP_03_Transit_Time_Delay.csv


Saving Gold_DS_SHIP_03_Transit_Time_Delay.csv to Gold_DS_SHIP_03_Transit_Time_Delay (1).csv
Saving Gold_DS_SHIP_02_Marine_Insurance.csv to Gold_DS_SHIP_02_Marine_Insurance (1).csv
Saving Gold_DS_SHIP_01_Freight_Rates.csv to Gold_DS_SHIP_01_Freight_Rates (1).csv
DATA QUALITY REPORT
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 16 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   date                      24 non-null     datetime64[ns]
 1   year                      24 non-null     int32         
 2   month                     24 non-null     object        
 3   month_no                  24 non-null     int32         
 4   quarter                   24 non-null     object        
 5   route                     15 non-null     object        
 6   freight_rate_usd_per_day  15 non-null     float64       
 7   war_risk_premium_percent  24 non-null     float64   

/tmp/ipykernel_2337/2883888027.py:102: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df["freight_rate_usd_per_day"].pct_change()*100


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# Hormuz Impact Project
# ETL Module : Inflation
# Dataset    : DS-INF-01_Consumer_Price_Index.csv
# Output     : Gold_DS_INF_01_Consumer_Price_Index.csv
# Version    : 1.0
# ============================================================

import pandas as pd
from google.colab import files

# ============================================================
# STEP 1 : Upload
# ============================================================

uploaded = files.upload()
filename = list(uploaded.keys())[0]

# ============================================================
# STEP 2 : Read Dataset
# ============================================================

df = pd.read_csv(filename)

print("="*70)
print("ORIGINAL DATASET")
print("="*70)
print(df.head())

# ============================================================
# STEP 3 : Date
# ============================================================

df["date"] = pd.to_datetime(df["date"])

df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month_name()
df["month_no"] = df["date"].dt.month
df["quarter"] = "Q" + df["date"].dt.quarter.astype(str)

# ============================================================
# STEP 4 : Numeric Conversion
# ============================================================

numeric_cols = [
    "cpi_index",
    "cpi_inflation"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# ============================================================
# STEP 5 : Sort
# ============================================================

df = df.sort_values("date").reset_index(drop=True)

# ============================================================
# STEP 6 : Derived KPIs
# ============================================================

df["monthly_cpi_change"] = (
    df["cpi_index"].diff()
).round(2)

df["monthly_cpi_change_pct"] = (
    df["cpi_index"].pct_change() * 100
).round(2)

# ============================================================
# STEP 7 : Remove Duplicates
# ============================================================

df.drop_duplicates(inplace=True)

# ============================================================
# STEP 8 : Final Columns
# ============================================================

df = df[
    [
        "date",
        "year",
        "month",
        "month_no",
        "quarter",
        "cpi_index",
        "cpi_inflation",
        "monthly_cpi_change",
        "monthly_cpi_change_pct"
    ]
]

# ============================================================
# STEP 9 : Validation
# ============================================================

print("\n" + "="*70)
print("DATA QUALITY REPORT")
print("="*70)

print(df.info())

print("\nMissing Values")
print(df.isnull().sum())

print("\nDuplicate Rows")
print(df.duplicated().sum())

print("\nSummary Statistics")
print(df.describe())

# ============================================================
# STEP 10 : Export
# ============================================================

output = "Gold_DS_INF_01_Consumer_Price_Index.csv"

df.to_csv(output, index=False)

print("\nGold Dataset Created Successfully!")

# ============================================================
# STEP 11 : Download
# ============================================================

files.download(output)

Saving DS-INF-01_Consumer_Price_Index.csv to DS-INF-01_Consumer_Price_Index.csv
ORIGINAL DATASET
         date  year     month  cpi_index  cpi_inflation
0  2025-01-01  2025   January     101.67            NaN
1  2025-02-01  2025  February     101.32          -0.34
2  2025-03-01  2025     March     101.39           0.07
3  2025-04-01  2025     April     101.58           0.19
4  2025-05-01  2025       May     101.90           0.32

DATA QUALITY REPORT
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17 entries, 0 to 16
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   date                    17 non-null     datetime64[ns]
 1   year                    17 non-null     int32         
 2   month                   17 non-null     object        
 3   month_no                17 non-null     int32         
 4   quarter                 17 non-null     object        
 5   cpi_index          

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# Hormuz Impact Project
# ETL Module : Inflation
# Dataset    : DS-INF-02_Wholesale_Price_Index.csv
# Output     : Gold_DS_INF_02_Wholesale_Price_Index.csv
# Version    : 1.0
# ============================================================

import pandas as pd
from google.colab import files

# ============================================================
# STEP 1 : Upload
# ============================================================

uploaded = files.upload()
filename = list(uploaded.keys())[0]

# ============================================================
# STEP 2 : Read Dataset
# ============================================================

df = pd.read_csv(filename)

print("=" * 70)
print("ORIGINAL DATASET")
print("=" * 70)
print(df.head())

# ============================================================
# STEP 3 : Date Processing
# ============================================================

df["date"] = pd.to_datetime(df["date"])

df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month_name()
df["month_no"] = df["date"].dt.month
df["quarter"] = "Q" + df["date"].dt.quarter.astype(str)

# ============================================================
# STEP 4 : Numeric Conversion
# ============================================================

df["wpi_index"] = pd.to_numeric(
    df["wpi_index"],
    errors="coerce"
)

# ============================================================
# STEP 5 : Sort
# ============================================================

df = df.sort_values("date").reset_index(drop=True)

# ============================================================
# STEP 6 : Derived KPIs
# ============================================================

df["monthly_wpi_change"] = (
    df["wpi_index"].diff()
).round(2)

df["monthly_wpi_change_pct"] = (
    df["wpi_index"].pct_change() * 100
).round(2)

# ============================================================
# STEP 7 : Remove Duplicates
# ============================================================

df.drop_duplicates(inplace=True)

# ============================================================
# STEP 8 : Final Columns
# ============================================================

df = df[
    [
        "date",
        "year",
        "month",
        "month_no",
        "quarter",
        "wpi_index",
        "monthly_wpi_change",
        "monthly_wpi_change_pct"
    ]
]

# ============================================================
# STEP 9 : Validation
# ============================================================

print("\n" + "=" * 70)
print("DATA QUALITY REPORT")
print("=" * 70)

print(df.info())

print("\nMissing Values")
print(df.isnull().sum())

print("\nDuplicate Rows")
print(df.duplicated().sum())

print("\nSummary Statistics")
print(df.describe())

# ============================================================
# STEP 10 : Export
# ============================================================

output = "Gold_DS_INF_02_Wholesale_Price_Index.csv"

df.to_csv(output, index=False)

print("\nGold Dataset Created Successfully!")

# ============================================================
# STEP 11 : Download
# ============================================================

files.download(output)

Saving DS-INF-02_Wholesale_Price_Index.csv to DS-INF-02_Wholesale_Price_Index.csv
ORIGINAL DATASET
         date  year   month  wpi_index
0  2023-04-01  2023   April       99.0
1  2023-05-01  2023     May       98.6
2  2023-06-01  2023    June       98.3
3  2023-07-01  2023    July       99.1
4  2023-08-01  2023  August       99.5

DATA QUALITY REPORT
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38 entries, 0 to 37
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   date                    38 non-null     datetime64[ns]
 1   year                    38 non-null     int32         
 2   month                   38 non-null     object        
 3   month_no                38 non-null     int32         
 4   quarter                 38 non-null     object        
 5   wpi_index               38 non-null     float64       
 6   monthly_wpi_change      37 non-null     float64       
 7  

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# Hormuz Impact Project
# ETL Module : Inflation
# Dataset    : DS-INF-03_Housing_Energy_CPI.csv
# Output     : Gold_DS_INF_03_Housing_Energy_CPI.csv
# Version    : 1.0
# ============================================================

import pandas as pd
from google.colab import files

# ============================================================
# STEP 1 : Upload
# ============================================================

uploaded = files.upload()
filename = list(uploaded.keys())[0]

# ============================================================
# STEP 2 : Read Dataset
# ============================================================

df = pd.read_csv(filename)

print("="*70)
print("ORIGINAL DATASET")
print("="*70)
print(df.head())

# ============================================================
# STEP 3 : Date Processing
# ============================================================

df["date"] = pd.to_datetime(df["date"])

df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month_name()
df["month_no"] = df["date"].dt.month
df["quarter"] = "Q" + df["date"].dt.quarter.astype(str)

# ============================================================
# STEP 4 : Numeric Conversion
# ============================================================

numeric_cols = [
    "housing_energy_index",
    "housing_energy_inflation"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# ============================================================
# STEP 5 : Sort
# ============================================================

df = df.sort_values("date").reset_index(drop=True)

# ============================================================
# STEP 6 : Derived KPIs
# ============================================================

df["monthly_housing_energy_change"] = (
    df["housing_energy_index"].diff()
).round(2)

df["monthly_housing_energy_change_pct"] = (
    df["housing_energy_index"].pct_change() * 100
).round(2)

# ============================================================
# STEP 7 : Remove Duplicates
# ============================================================

df.drop_duplicates(inplace=True)

# ============================================================
# STEP 8 : Final Columns
# ============================================================

df = df[
    [
        "date",
        "year",
        "month",
        "month_no",
        "quarter",
        "housing_energy_index",
        "housing_energy_inflation",
        "monthly_housing_energy_change",
        "monthly_housing_energy_change_pct"
    ]
]

# ============================================================
# STEP 9 : Validation
# ============================================================

print("\n" + "="*70)
print("DATA QUALITY REPORT")
print("="*70)

print(df.info())

print("\nMissing Values")
print(df.isnull().sum())

print("\nDuplicate Rows")
print(df.duplicated().sum())

print("\nSummary Statistics")
print(df.describe())

# ============================================================
# STEP 10 : Export
# ============================================================

output = "Gold_DS_INF_03_Housing_Energy_CPI.csv"

df.to_csv(output, index=False)

print("\nGold Dataset Created Successfully!")

# ============================================================
# STEP 11 : Download
# ============================================================

files.download(output)

Saving DS-INF-03_Housing_Energy_CPI.csv to DS-INF-03_Housing_Energy_CPI.csv
ORIGINAL DATASET
         date  year     month  housing_energy_index  housing_energy_inflation
0  2025-01-01  2025   January                100.48                       NaN
1  2025-02-01  2025  February                100.56                      0.08
2  2025-03-01  2025     March                100.71                      0.15
3  2025-04-01  2025     April                101.11                      0.40
4  2025-05-01  2025       May                101.35                      0.24

DATA QUALITY REPORT
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17 entries, 0 to 16
Data columns (total 9 columns):
 #   Column                             Non-Null Count  Dtype         
---  ------                             --------------  -----         
 0   date                               17 non-null     datetime64[ns]
 1   year                               17 non-null     int32         
 2   month                      

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# Hormuz Impact Project
# ETL Module : Inflation
# Dataset    : DS-INF-04_Food_Beverages_CPI.csv
# Output     : Gold_DS_INF_04_Food_Beverages_CPI.csv
# Version    : 1.0
# ============================================================

import pandas as pd
from google.colab import files

# ============================================================
# STEP 1 : Upload
# ============================================================

uploaded = files.upload()
filename = list(uploaded.keys())[0]

# ============================================================
# STEP 2 : Read Dataset
# ============================================================

df = pd.read_csv(filename)

print("=" * 70)
print("ORIGINAL DATASET")
print("=" * 70)
print(df.head())

# ============================================================
# STEP 3 : Date Processing
# ============================================================

df["date"] = pd.to_datetime(df["date"])

df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month_name()
df["month_no"] = df["date"].dt.month
df["quarter"] = "Q" + df["date"].dt.quarter.astype(str)

# ============================================================
# STEP 4 : Numeric Conversion
# ============================================================

numeric_cols = [
    "food_beverages_index",
    "food_beverages_inflation"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# ============================================================
# STEP 5 : Sort
# ============================================================

df = df.sort_values("date").reset_index(drop=True)

# ============================================================
# STEP 6 : Derived KPIs
# ============================================================

df["monthly_food_beverages_change"] = (
    df["food_beverages_index"].diff()
).round(2)

df["monthly_food_beverages_change_pct"] = (
    df["food_beverages_index"].pct_change() * 100
).round(2)

# ============================================================
# STEP 7 : Remove Duplicates
# ============================================================

df.drop_duplicates(inplace=True)

# ============================================================
# STEP 8 : Final Columns
# ============================================================

df = df[
    [
        "date",
        "year",
        "month",
        "month_no",
        "quarter",
        "food_beverages_index",
        "food_beverages_inflation",
        "monthly_food_beverages_change",
        "monthly_food_beverages_change_pct"
    ]
]

# ============================================================
# STEP 9 : Validation
# ============================================================

print("\n" + "=" * 70)
print("DATA QUALITY REPORT")
print("=" * 70)

print(df.info())

print("\nMissing Values")
print(df.isnull().sum())

print("\nDuplicate Rows")
print(df.duplicated().sum())

print("\nSummary Statistics")
print(df.describe())

# ============================================================
# STEP 10 : Export
# ============================================================

output = "Gold_DS_INF_04_Food_Beverages_CPI.csv"

df.to_csv(output, index=False)

print("\nGold Dataset Created Successfully!")

# ============================================================
# STEP 11 : Download
# ============================================================

files.download(output)

Saving DS-INF-04_Food_Beverages_CPI.csv to DS-INF-04_Food_Beverages_CPI.csv
ORIGINAL DATASET
         date  year     month  food_beverages_index  food_beverages_inflation
0  2025-01-01  2025   January                101.87                       NaN
1  2025-02-01  2025  February                100.50                     -1.34
2  2025-03-01  2025     March                100.40                     -0.10
3  2025-04-01  2025     April                100.33                     -0.07
4  2025-05-01  2025       May                100.69                      0.36

DATA QUALITY REPORT
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17 entries, 0 to 16
Data columns (total 9 columns):
 #   Column                             Non-Null Count  Dtype         
---  ------                             --------------  -----         
 0   date                               17 non-null     datetime64[ns]
 1   year                               17 non-null     int32         
 2   month                      

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# Hormuz Impact Project
# ETL Module : Inflation
# Dataset    : DS-INF-05_Transport_CPI.csv
# Output     : Gold_DS_INF_05_Transport_CPI.csv
# Version    : 1.0
# ============================================================

import pandas as pd
from google.colab import files

# ============================================================
# STEP 1 : Upload Dataset
# ============================================================

uploaded = files.upload()
filename = list(uploaded.keys())[0]

# ============================================================
# STEP 2 : Read Dataset
# ============================================================

df = pd.read_csv(filename)

print("="*70)
print("ORIGINAL DATASET")
print("="*70)

print(df.head())

# ============================================================
# STEP 3 : Date Processing
# ============================================================

df["date"] = pd.to_datetime(df["date"])

df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month_name()
df["month_no"] = df["date"].dt.month
df["quarter"] = "Q" + df["date"].dt.quarter.astype(str)

# ============================================================
# STEP 4 : Numeric Conversion
# ============================================================

numeric_cols = [
    "transport_index",
    "transport_inflation"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# ============================================================
# STEP 5 : Sort
# ============================================================

df = df.sort_values("date").reset_index(drop=True)

# ============================================================
# STEP 6 : Derived KPIs
# ============================================================

df["monthly_transport_change"] = (
    df["transport_index"].diff()
).round(2)

df["monthly_transport_change_pct"] = (
    df["transport_index"].pct_change() * 100
).round(2)

# ============================================================
# STEP 7 : Remove Duplicates
# ============================================================

df.drop_duplicates(inplace=True)

# ============================================================
# STEP 8 : Final Columns
# ============================================================

df = df[
    [
        "date",
        "year",
        "month",
        "month_no",
        "quarter",
        "transport_index",
        "transport_inflation",
        "monthly_transport_change",
        "monthly_transport_change_pct"
    ]
]

# ============================================================
# STEP 9 : Validation
# ============================================================

print("\n" + "="*70)
print("DATA QUALITY REPORT")
print("="*70)

print(df.info())

print("\nMissing Values")
print(df.isnull().sum())

print("\nDuplicate Rows")
print(df.duplicated().sum())

print("\nSummary Statistics")
print(df.describe())

# ============================================================
# STEP 10 : Export
# ============================================================

output = "Gold_DS_INF_05_Transport_CPI.csv"

df.to_csv(output, index=False)

print("\nGold Dataset Created Successfully!")

# ============================================================
# STEP 11 : Download
# ============================================================

files.download(output)

Saving DS-INF-05_Transport_CPI.csv to DS-INF-05_Transport_CPI.csv
ORIGINAL DATASET
         date  year     month  transport_index  transport_inflation
0  2025-01-01  2025   January           100.55                  NaN
1  2025-02-01  2025  February           100.74                 0.19
2  2025-03-01  2025     March           100.73                -0.01
3  2025-04-01  2025     April           100.85                 0.12
4  2025-05-01  2025       May           100.96                 0.11

DATA QUALITY REPORT
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17 entries, 0 to 16
Data columns (total 9 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   date                          17 non-null     datetime64[ns]
 1   year                          17 non-null     int32         
 2   month                         17 non-null     object        
 3   month_no                      17 non-null     int32

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# Hormuz Impact Project
# ADS Module : Inflation
# Dataset    : ADS_INF_01_Inflation_Analysis.csv
# Version    : 1.0
# ============================================================

import pandas as pd
import numpy as np
from google.colab import files

# ============================================================
# STEP 1 : Upload Gold Datasets
# ============================================================

print("Upload the following Gold datasets:")
print("1. Gold_DS_INF_01_Consumer_Price_Index.csv")
print("2. Gold_DS_INF_02_Wholesale_Price_Index.csv")
print("3. Gold_DS_INF_03_Housing_Energy_CPI.csv")
print("4. Gold_DS_INF_04_Food_Beverages_CPI.csv")
print("5. Gold_DS_INF_05_Transport_CPI.csv")
print("6. Gold_DS_FX_03_RBI_Repo_Rate.csv")

uploaded = files.upload()

# ============================================================
# STEP 2 : Read Files
# ============================================================

cpi = None
wpi = None
housing = None
food = None
transport = None
repo = None

for file in uploaded.keys():

    if "INF_01" in file:
        cpi = pd.read_csv(file)

    elif "INF_02" in file:
        wpi = pd.read_csv(file)

    elif "INF_03" in file:
        housing = pd.read_csv(file)

    elif "INF_04" in file:
        food = pd.read_csv(file)

    elif "INF_05" in file:
        transport = pd.read_csv(file)

    elif "FX_03" in file:
        repo = pd.read_csv(file)

# ============================================================
# STEP 3 : Date Conversion
# ============================================================

for df in [cpi, wpi, housing, food, transport, repo]:
    df["date"] = pd.to_datetime(df["date"])

# ============================================================
# STEP 4 : Keep Required Columns
# ============================================================

cpi = cpi[
    [
        "date",
        "cpi_index",
        "cpi_inflation"
    ]
]

wpi = wpi[
    [
        "date",
        "wpi_index"
    ]
]

housing = housing[
    [
        "date",
        "housing_energy_index"
    ]
]

food = food[
    [
        "date",
        "food_beverages_index"
    ]
]

transport = transport[
    [
        "date",
        "transport_index"
    ]
]

repo = repo[
    [
        "date",
        "repo_rate"
    ]
]

# ============================================================
# STEP 5 : Merge
# ============================================================

df = cpi.merge(
    wpi,
    on="date",
    how="left"
)

df = df.merge(
    housing,
    on="date",
    how="left"
)

df = df.merge(
    food,
    on="date",
    how="left"
)

df = df.merge(
    transport,
    on="date",
    how="left"
)

df = df.merge(
    repo,
    on="date",
    how="left"
)

# ============================================================
# STEP 6 : Sort & Calendar
# ============================================================

df = df.sort_values("date").reset_index(drop=True)

df["repo_rate"] = df["repo_rate"].ffill()

df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month_name()
df["month_no"] = df["date"].dt.month
df["quarter"] = "Q" + df["date"].dt.quarter.astype(str)

# ============================================================
# STEP 7 : Derived KPIs
# ============================================================

df["food_weight_pct"] = (
    df["food_beverages_index"] /
    df["cpi_index"] * 100
).round(2)

df["housing_weight_pct"] = (
    df["housing_energy_index"] /
    df["cpi_index"] * 100
).round(2)

df["transport_weight_pct"] = (
    df["transport_index"] /
    df["cpi_index"] * 100
).round(2)

# ============================================================
# STEP 8 : Inflation Pressure Score
# ============================================================

def normalize(series):
    series = series.ffill()

    if series.max() == series.min():
        return pd.Series(50, index=series.index)

    return (
        (series - series.min()) /
        (series.max() - series.min())
    ) * 100

food_norm = normalize(df["food_beverages_index"])
housing_norm = normalize(df["housing_energy_index"])
transport_norm = normalize(df["transport_index"])
cpi_norm = normalize(df["cpi_inflation"].fillna(0))

df["inflation_pressure_score"] = (
      food_norm * 0.35
    + housing_norm * 0.25
    + transport_norm * 0.20
    + cpi_norm * 0.20
).round(2)

# ============================================================
# STEP 9 : Inflation Risk Level
# ============================================================

conditions = [
    df["inflation_pressure_score"] < 35,
    (df["inflation_pressure_score"] >= 35) &
    (df["inflation_pressure_score"] < 70),
    df["inflation_pressure_score"] >= 70
]

choices = [
    "Low",
    "Moderate",
    "High"
]

df["inflation_risk_level"] = np.select(
    conditions,
    choices,
    default="Moderate"
)

# ============================================================
# STEP 10 : Final Columns
# ============================================================

df = df[
    [
        "date",
        "year",
        "month",
        "month_no",
        "quarter",

        "cpi_index",
        "cpi_inflation",

        "wpi_index",

        "housing_energy_index",
        "food_beverages_index",
        "transport_index",

        "repo_rate",

        "food_weight_pct",
        "housing_weight_pct",
        "transport_weight_pct",

        "inflation_pressure_score",
        "inflation_risk_level"
    ]
]

# ============================================================
# STEP 11 : Validation
# ============================================================

print("=" * 70)
print("DATA QUALITY REPORT")
print("=" * 70)

print(df.info())

print("\nMissing Values")
print(df.isnull().sum())

print("\nDuplicate Rows")
print(df.duplicated().sum())

print("\nSummary Statistics")
print(df.describe())

# ============================================================
# STEP 12 : Export
# ============================================================

output = "ADS_INF_01_Inflation_Analysis.csv"

df.to_csv(output, index=False)

print("\nADS Dataset Created Successfully!")

files.download(output)

Upload the following Gold datasets:
1. Gold_DS_INF_01_Consumer_Price_Index.csv
2. Gold_DS_INF_02_Wholesale_Price_Index.csv
3. Gold_DS_INF_03_Housing_Energy_CPI.csv
4. Gold_DS_INF_04_Food_Beverages_CPI.csv
5. Gold_DS_INF_05_Transport_CPI.csv
6. Gold_DS_FX_03_RBI_Repo_Rate.csv


Saving Gold_DS_INF_05_Transport_CPI.csv to Gold_DS_INF_05_Transport_CPI (1).csv
Saving Gold_DS_INF_04_Food_Beverages_CPI.csv to Gold_DS_INF_04_Food_Beverages_CPI (1).csv
Saving Gold_DS_INF_03_Housing_Energy_CPI.csv to Gold_DS_INF_03_Housing_Energy_CPI (1).csv
Saving Gold_DS_INF_02_Wholesale_Price_Index.csv to Gold_DS_INF_02_Wholesale_Price_Index (1).csv
Saving Gold_DS_INF_01_Consumer_Price_Index.csv to Gold_DS_INF_01_Consumer_Price_Index (1).csv
Saving Gold_DS_FX_03_RBI_Repo_Rate.csv to Gold_DS_FX_03_RBI_Repo_Rate (2).csv
DATA QUALITY REPORT
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17 entries, 0 to 16
Data columns (total 17 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   date                      17 non-null     datetime64[ns]
 1   year                      17 non-null     int32         
 2   month                     17 non-null     object        
 3   month_no                  17 non-n

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ============================================================
# Dashboard Layer
# Dashboard_Oil.csv
# ============================================================

import pandas as pd
from google.colab import files

# ============================================================
# Upload Files
# ============================================================

uploaded = files.upload()

# ============================================================
# Read Files
# ============================================================

brent = pd.read_csv("Gold_DS_OIL_01_Brent_Crude.csv")
imports = pd.read_csv("Gold_DS_OIL_02_Crude_Oil_Imports.csv")
fuel = pd.read_csv("Gold_DS_OIL_03_Petrol_Diesel_Prices.csv")
production = pd.read_csv("Gold_DS_OIL_04_Domestic_Crude_Production.csv")

# ============================================================
# Date Conversion
# ============================================================

brent["date"] = pd.to_datetime(brent["date"])
imports["date"] = pd.to_datetime(imports["date"])
fuel["date"] = pd.to_datetime(fuel["date"])

# ============================================================
# Monthly Brent Average
# ============================================================

brent_monthly = (
    brent
    .groupby(["year", "month_no", "month"], as_index=False)
    .agg(
        avg_brent_price=("close_price", "mean"),
        max_brent_price=("high_price", "max"),
        min_brent_price=("low_price", "min")
    )
)

# ============================================================
# Monthly Petrol/Diesel Average
# ============================================================

fuel["year"] = fuel["date"].dt.year
fuel["month_no"] = fuel["date"].dt.month
fuel["month"] = fuel["date"].dt.month_name()

fuel_monthly = (
    fuel
    .groupby(["year", "month_no", "month"], as_index=False)
    .agg(
        avg_petrol_price=("petrol_price", "mean"),
        avg_diesel_price=("diesel_price", "mean")
    )
)

# ============================================================
# Monthly Imports
# ============================================================

imports_monthly = imports[
    [
        "year",
        "month_no",
        "month",
        "quantity_000_mt",
        "value_usd_million",
        "value_inr_crore",
        "avg_import_price_usd_per_mt",
        "avg_import_price_inr_per_mt"
    ]
]

# ============================================================
# Domestic Production
# ============================================================

production = production.rename(
    columns={
        "crude_oil_production_000_mt": "domestic_production_000_mt"
    }
)

# Keep only useful columns if they exist
prod_cols = [c for c in [
    "year",
    "domestic_production_000_mt"
] if c in production.columns]

production = production[prod_cols]

# ============================================================
# Merge
# ============================================================

dashboard = brent_monthly.merge(
    fuel_monthly,
    on=["year", "month_no", "month"],
    how="left"
)

dashboard = dashboard.merge(
    imports_monthly,
    on=["year", "month_no", "month"],
    how="left"
)

dashboard = dashboard.merge(
    production,
    on="year",
    how="left"
)

# ============================================================
# Date Column
# ============================================================

dashboard["date"] = pd.to_datetime(
    dashboard["year"].astype(str)
    + "-"
    + dashboard["month_no"].astype(str)
    + "-01"
)

dashboard["quarter"] = "Q" + dashboard["date"].dt.quarter.astype(str)

# ============================================================
# Round Numeric Columns
# ============================================================

numeric_cols = dashboard.select_dtypes(include="number").columns

dashboard[numeric_cols] = dashboard[numeric_cols].round(2)

# ============================================================
# Arrange Columns
# ============================================================

cols = [
    "date",
    "year",
    "month",
    "month_no",
    "quarter",

    "avg_brent_price",
    "max_brent_price",
    "min_brent_price",

    "avg_petrol_price",
    "avg_diesel_price",

    "quantity_000_mt",
    "value_usd_million",
    "value_inr_crore",

    "avg_import_price_usd_per_mt",
    "avg_import_price_inr_per_mt",

    "domestic_production_000_mt"
]

dashboard = dashboard[[c for c in cols if c in dashboard.columns]]

dashboard = dashboard.sort_values("date").reset_index(drop=True)

# ============================================================
# Export
# ============================================================

dashboard.to_csv("Dashboard_Oil.csv", index=False)

print("="*60)
print("Dashboard_Oil.csv Created Successfully")
print("="*60)

print(dashboard.head())

files.download("Dashboard_Oil.csv")

Saving Gold_DS_OIL_05_Country_Wise_Import_Share.csv to Gold_DS_OIL_05_Country_Wise_Import_Share.csv
Saving Gold_DS_OIL_04_Domestic_Crude_Production.csv to Gold_DS_OIL_04_Domestic_Crude_Production.csv
Saving Gold_DS_OIL_03_Petrol_Diesel_Prices.csv to Gold_DS_OIL_03_Petrol_Diesel_Prices.csv
Saving Gold_DS_OIL_02_Crude_Oil_Imports.csv to Gold_DS_OIL_02_Crude_Oil_Imports.csv
Saving Gold_DS_OIL_01_Brent_Crude.csv to Gold_DS_OIL_01_Brent_Crude.csv
Dashboard_Oil.csv Created Successfully
        date  year     month  month_no quarter  avg_brent_price  \
0 2024-01-01  2024   January         1      Q1            79.20   
1 2024-02-01  2024  February         2      Q1            81.62   
2 2024-03-01  2024     March         3      Q1            84.67   
3 2024-04-01  2024     April         4      Q2            89.00   
4 2024-05-01  2024       May         5      Q2            82.99   

   max_brent_price  min_brent_price  avg_petrol_price  avg_diesel_price  \
0            84.79            74.79  

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>